# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.28.21.1.6.2 — FAST
## Aphys Closure and Direction-Global Phase-Space Principal Symbol

### Verrou unique

Cette étape prend comme parent canonique `.28.21.1.6.1`, qui a fermé le pont principal Hamilton–Dirac gauge-préservant :

\[
\boxed{
\mathbb A_{\rm HD}(\mathbf n)\in\mathbb R^{28\times28},
\qquad
C_{\rm phys}\mathbb A_{\rm HD}R_{20}=0.
}
\]

La mission unique est :

\[
\boxed{
\mathbb A_{\rm HD}
+
R_{20}^{(\alpha)}
\longrightarrow
\widehat{\mathbb A}_{\rm phys}^{(\alpha)}(\hat{\mathbf k})
\longrightarrow
\text{transition/similarity audit}.
}
\]

Aucune diagonalisation, aucun spectre, aucun symétriseur et aucun verdict d'hyperbolicité forte n'est autorisé ici.

### Parent exécuté/audité

`.28.21.1.6.1` :

- taille : `60240` octets ;
- SHA-256 : `e759763ba3f2f30b9fe95f96fa9e14a177ea46ad9002ecf9270ae451be0eb8a8` ;
- `SOURCE_EXACT=True` ;
- `CANONICAL_BRIDGE_PRINCIPAL_GATE_PASS=True` ;
- `HAMILTON_DIRAC_GAUGE_PRESERVING_FLOW_PASS=True` ;
- `FULL_A_PHYS_MATERIALIZED=False`.

### Scope lock

Le fond reste le témoin sain anisotrope spectral-diagonal à coefficients gelés :

\[
(a_0,a_1,a_2,a_3)
=
\left(\frac34,-\frac15,-\frac14,-\frac3{10}\right),
\qquad
K_S=1,
\quad
\kappa_D=2,
\quad
M_{\rm Pl}^2=1.
\]

`FULL_A_PHYS_MATERIALIZED=True` dans ce notebook signifiera **complet sur toutes les directions spatiales non nulles pour ce fond sain fixé**, et non encore sur tout l'espace des fonds/paramètres GVH.

In [1]:
from __future__ import annotations

import sys, json, hashlib, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

PARENT_2821161 = {
    "version":"0.3.2.7.3.7.3.3.28.21.1.6.1",
    "executed_size_bytes":60240,
    "executed_sha256":
        "e759763ba3f2f30b9fe95f96fa9e14a177ea46ad9002ecf9270ae451be0eb8a8",
    "source_exact":True,
    "canonical_bridge_principal_gate_pass":True,
    "Hamilton_Dirac_gauge_preserving_flow_pass":True,
    "full_A_phys_materialized":False,
}

G2821162_PROVENANCE_GATE_PASS = all([
    PARENT_2821161["source_exact"],
    PARENT_2821161["canonical_bridge_principal_gate_pass"],
    PARENT_2821161["Hamilton_Dirac_gauge_preserving_flow_pass"],
    not PARENT_2821161["full_A_phys_materialized"],
])

assert G2821162_PROVENANCE_GATE_PASS

print("Python =",sys.version.split()[0])
print("SymPy =",sp.__version__)
print("NumPy =",np.__version__)
print("G2821162_PROVENANCE_GATE_PASS =",G2821162_PROVENANCE_GATE_PASS)

Python = 3.13.15
SymPy = 1.14.0
NumPy = 2.1.3
G2821162_PROVENANCE_GATE_PASS = True


# Niveau 1 — Reconstruction autonome de \(K,M,G\)

Le notebook reconstruit le même principal Pure-GVH-P que `.28.21.1.4 → .1.6.1` :

\[
P_{\rm raw}(\omega,\mathbf k)
=
\omega^2K
+
\omega k_iM^i
+
k_i k_jG^{ij}.
\]

Cela évite toute dépendance d'exécution à un fichier parent externe.

In [2]:

eta=sp.diag(-1,1,1,1)

a0,a1,a2=sp.symbols("a0 a1 a2",real=True)
a3=-a0-a1-a2

Abar=sp.diag(a0,a1,a2,a3)
Qbar=sp.factor(sp.trace(Abar*Abar))

KS,kappaD,Mpl2=sp.symbols(
    "K_S kappa_D Mpl2",
    real=True
)

names=[
    "n","beta1","beta2","beta3",
    "h11","h22","h33","h12","h13","h23",
    "D00","D01","D02","D03",
    "D11","D22","D33","D12","D13","D23",
]

h_basis=[]
D_basis=[]

for name in names:
    h=sp.zeros(4)
    d=sp.zeros(4)

    if name=="n":
        h[0,0]=-2
    elif name.startswith("beta"):
        i=int(name[-1])
        h[0,i]=h[i,0]=1
    elif name.startswith("h"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        h[i,j]=h[j,i]=1
    elif name.startswith("D"):
        ij=name[1:]
        i,j=int(ij[0]),int(ij[1])
        d[i,j]=d[j,i]=1

    h_basis.append(h)
    D_basis.append(d)

dA_basis=[]

for h,d in zip(h_basis,D_basis):
    dM=-eta*h*Abar+eta*d
    dA=dM-sp.trace(dM)*sp.eye(4)/4
    dA_basis.append(dA)

def build_principal_matrix(p):
    H=sp.zeros(20)

    # Pure-GVH-P sector
    for alpha_idx in range(4):
        B=[]
        J=[]
        C=[]

        for h,dA in zip(h_basis,dA_basis):
            Gamma=sp.zeros(4)

            for mu in range(4):
                for nu in range(4):
                    acc=0
                    for rho in range(4):
                        acc += eta[mu,rho]*(
                            p[alpha_idx]*h[rho,nu]
                            +p[nu]*h[rho,alpha_idx]
                            -p[rho]*h[alpha_idx,nu]
                        )/2
                    Gamma[mu,nu]=acc

            Bj=p[alpha_idx]*dA+Gamma*Abar-Abar*Gamma
            B.append(Bj)
            J.append(sp.trace(Abar*Bj))
            C.append(Abar*Bj-Bj*Abar)

        sign=eta[alpha_idx,alpha_idx]

        for j in range(20):
            for k in range(j,20):
                shape=Qbar*sp.trace(B[j]*B[k])-J[j]*J[k]
                angle=sp.trace(C[j]*C[k])

                val=(
                    -KS*sign*shape
                    -kappaD*sp.Rational(1,2)*sign*angle
                )

                H[j,k]+=val
                if k!=j:
                    H[k,j]+=val

    # Einstein-Hilbert / Fierz-Pauli benchmark
    pvec=sp.Matrix(p)
    pup=eta*pvec
    p2=(pvec.T*eta*pvec)[0]

    attrs=[]

    for h in h_basis[:10]:
        hup=eta*h*eta
        v=[
            sum(p[mu]*hup[mu,nu] for mu in range(4))
            for nu in range(4)
        ]
        w=[
            sum(pup[lam]*h[lam,nu] for lam in range(4))
            for nu in range(4)
        ]
        trh=sp.trace(eta*h)
        vp=sum(v[nu]*p[nu] for nu in range(4))
        attrs.append((h,hup,v,w,trh,vp))

    for j in range(10):
        hj,hjup,vj,wj,trj,vpj=attrs[j]

        for k in range(j,10):
            hk,hkup,vk,wk,trk,vpk=attrs[k]

            inner=sum(
                hj[mu,nu]*hkup[mu,nu]
                for mu in range(4)
                for nu in range(4)
            )

            BF=(
                p2*inner
                -sum(
                    vj[nu]*wk[nu]+vk[nu]*wj[nu]
                    for nu in range(4)
                )
                +vpj*trk
                +vpk*trj
                -p2*trj*trk
            )

            val=-Mpl2*sp.Rational(1,4)*BF

            H[j,k]+=val
            if k!=j:
                H[k,j]+=val

    return H

e0=(1,0,0,0)
e1=(0,1,0,0)
e2=(0,0,1,0)
e3=(0,0,0,1)

P_e0=build_principal_matrix(e0)
P_e1=build_principal_matrix(e1)
P_e2=build_principal_matrix(e2)
P_e3=build_principal_matrix(e3)

K_raw=P_e0

M_raw={
    1:build_principal_matrix((1,1,0,0))-P_e0-P_e1,
    2:build_principal_matrix((1,0,1,0))-P_e0-P_e2,
    3:build_principal_matrix((1,0,0,1))-P_e0-P_e3,
}

G_raw={
    (1,1):P_e1,
    (2,2):P_e2,
    (3,3):P_e3,
    (1,2):(build_principal_matrix((0,1,1,0))-P_e1-P_e2)/2,
    (1,3):(build_principal_matrix((0,1,0,1))-P_e1-P_e3)/2,
    (2,3):(build_principal_matrix((0,0,1,1))-P_e2-P_e3)/2,
}

G2821161_RAW_PENCIL_RECONSTRUCTED=all([
    K_raw.shape==(20,20),
    all(M_raw[i].shape==(20,20) for i in (1,2,3)),
    all(G_raw[key].shape==(20,20) for key in G_raw),
])

assert G2821161_RAW_PENCIL_RECONSTRUCTED

print("G2821161_RAW_PENCIL_RECONSTRUCTED =",G2821161_RAW_PENCIL_RECONSTRUCTED)


G2821161_RAW_PENCIL_RECONSTRUCTED = True


In [3]:
D_raw={i:sp.simplify(M_raw[i]/2) for i in (1,2,3)}
assert all(sp.simplify(D_raw[i]+D_raw[i].T-M_raw[i])==sp.zeros(20) for i in (1,2,3))
print("Principal Legendre representative D_i=M_i/2 fixed")

Principal Legendre representative D_i=M_i/2 fixed


# Niveau 1B — Fond sain, base cinétique et générateurs originaux

On reprend la réduction cinétique exacte :

\[
R_{\rm kin}:\mathbb R^{14}\to\mathbb R^{20},
\qquad
K_{14}=R_{\rm kin}^TKR_{\rm kin},
\qquad
\operatorname{rank}K_{14}=14.
\]

Les quatre directions nulles de difféomorphisme sont conservées dans :

\[
N_{\rm diff}\in\mathbb R^{20\times4}.
\]


In [4]:

healthy_subs={
    a0:sp.Rational(3,4),
    a1:-sp.Rational(1,5),
    a2:-sp.Rational(1,4),
    KS:1,
    kappaD:2,
    Mpl2:1,
}

K_h=K_raw.subs(healthy_subs)
M_h={i:M_raw[i].subs(healthy_subs) for i in (1,2,3)}
D_h={i:D_raw[i].subs(healthy_subs) for i in (1,2,3)}
G_h={key:G_raw[key].subs(healthy_subs) for key in G_raw}

R_kin=sp.Matrix.hstack(*K_h.columnspace())
K14=sp.simplify(R_kin.T*K_h*R_kin)
K14_inv=K14.inv()

a=[a0,a1,a2,a3]

def original_gauge_vectors(p):
    vectors=[]

    for sigma in range(4):
        zeta=[0,0,0,0]
        zeta[sigma]=1

        hg=sp.zeros(4)
        dD=sp.zeros(4)

        for mu in range(4):
            for nu in range(4):
                hg[mu,nu]=(
                    p[mu]*zeta[nu]
                    +p[nu]*zeta[mu]
                )

                dD[mu,nu]=(
                    a[nu]*p[mu]*zeta[nu]
                    +a[mu]*p[nu]*zeta[mu]
                )

        vec=sp.zeros(20,1)

        vec[0]=-hg[0,0]/2
        vec[1]=hg[0,1]
        vec[2]=hg[0,2]
        vec[3]=hg[0,3]
        vec[4]=hg[1,1]
        vec[5]=hg[2,2]
        vec[6]=hg[3,3]
        vec[7]=hg[1,2]
        vec[8]=hg[1,3]
        vec[9]=hg[2,3]

        vals=[
            dD[0,0],dD[0,1],dD[0,2],dD[0,3],
            dD[1,1],dD[2,2],dD[3,3],
            dD[1,2],dD[1,3],dD[2,3],
        ]

        for j,val in enumerate(vals,start=10):
            vec[j]=val

        vectors.append(vec)

    trace=sp.zeros(20,1)
    trace[10]=-1
    trace[14]=1
    trace[15]=1
    trace[16]=1

    vectors.append(trace)

    return vectors

N_diff=sp.Matrix.hstack(
    *original_gauge_vectors((1,0,0,0))[:4]
).subs(healthy_subs)

N_trace=sp.zeros(20,1)
N_trace[10]=-1
N_trace[14]=1
N_trace[15]=1
N_trace[16]=1

N_radial=sp.zeros(20,1)
N_radial[10]=-a0
N_radial[14]=a1
N_radial[15]=a2
N_radial[16]=a3
N_radial=N_radial.subs(healthy_subs)

N6=sp.Matrix.hstack(
    N_diff,
    N_trace,
    N_radial,
)

T20=sp.Matrix.hstack(
    R_kin,
    N6,
)

G2821161_ORIGINAL_NULL_BASIS_PASS=all([
    K_h.rank()==14,
    R_kin.shape==(20,14),
    R_kin.rank()==14,
    N6.shape==(20,6),
    N6.rank()==6,
    K_h*N6==sp.zeros(20,6),
    T20.rank()==20,
])

assert G2821161_ORIGINAL_NULL_BASIS_PASS

print("rank K_h =",K_h.rank())
print("rank N6 =",N6.rank())
print("rank T20 =",T20.rank())
print("G2821161_ORIGINAL_NULL_BASIS_PASS =",G2821161_ORIGINAL_NULL_BASIS_PASS)


rank K_h = 14
rank N6 = 6
rank T20 = 20
G2821161_ORIGINAL_NULL_BASIS_PASS = True


# Niveau 1C — Objets directionnels et chaîne de Noether

Pour \(\mathbf n=(n_1,n_2,n_3)\neq0\), on définit :

\[
B(\mathbf n)=n_iM^i,
\qquad
D(\mathbf n)=\frac12B(\mathbf n),
\qquad
C(\mathbf n)=n_i n_jG^{ij}.
\]

Le générateur spatial de difféomorphisme \(G_0(\mathbf n)\) sera utilisé pour construire une jauge globale régulière.

In [5]:
n1,n2,n3=sp.symbols("n1 n2 n3",real=True)

B_symbolic=(
    n1*M_h[1]
    +n2*M_h[2]
    +n3*M_h[3]
)

C_symbolic=(
    n1**2*G_h[(1,1)]
    +n2**2*G_h[(2,2)]
    +n3**2*G_h[(3,3)]
    +2*n1*n2*G_h[(1,2)]
    +2*n1*n3*G_h[(1,3)]
    +2*n2*n3*G_h[(2,3)]
)

G0_symbolic=sp.Matrix.hstack(
    *original_gauge_vectors((0,n1,n2,n3))[:4]
).subs(healthy_subs)

G1_symbolic=N_diff

noether_checks=[
    sp.simplify(K_h*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(K_h*G0_symbolic+B_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(B_symbolic*G0_symbolic+C_symbolic*G1_symbolic)==sp.zeros(20,4),
    sp.simplify(C_symbolic*G0_symbolic)==sp.zeros(20,4),
]

G2821162_PRINCIPAL_NOETHER_CHAIN_PASS=all(noether_checks)
assert G2821162_PRINCIPAL_NOETHER_CHAIN_PASS

T20_inv=T20.inv()
K14_inv=K14.inv()

print("Noether checks =",noether_checks)

Noether checks = [True, True, True, True]


# Niveau 2 — Jauge de Gram globale

L'atlas \(U_x,U_y,U_z\) de `.28.21.1.5/.1.6.1` était correct, mais il n'est pas minimal.

Décomposons le générateur spatial dans la base :

\[
T_{20}=[R_{\rm kin},N_6].
\]

Sa partie active est :

\[
\boxed{
Q(\mathbf n)
=
\left[T_{20}^{-1}G_0(\mathbf n)\right]_{1:14}
\in\mathbb R^{14\times4}.
}
\]

On choisit la jauge :

\[
\boxed{
F_{\rm G}(\mathbf n)=Q(\mathbf n)^T,
\qquad
F_{\rm G}x=0.
}
\]

Le déterminant de Gram :

\[
\det(Q^TQ)
\]

est un polynôme homogène pair dont tous les coefficients non nuls sont strictement positifs. Donc :

\[
\boxed{
\det(Q^TQ)>0
\quad\forall\ \mathbf n\neq0.
}
\]

Cette jauge est donc globale sur \(S^2\).

In [6]:
Q_symbolic=sp.simplify(
    (T20_inv*G0_symbolic)[:14,:]
)
F_global_symbolic=sp.simplify(Q_symbolic.T)
Gram_global=sp.simplify(Q_symbolic.T*Q_symbolic)
det_Gram=sp.factor(Gram_global.det())

poly_Gram=sp.Poly(sp.expand(det_Gram),n1,n2,n3)
gram_terms=poly_Gram.terms()

gram_even_exponents=all(
    all(e%2==0 for e in monom)
    for monom,coeff in gram_terms
)
gram_positive_coeffs=all(
    bool(coeff>0)
    for monom,coeff in gram_terms
)

G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS=all([
    Q_symbolic.shape==(14,4),
    gram_even_exponents,
    gram_positive_coeffs,
    len(gram_terms)>0,
])

assert G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS

print("det Gram total degree =",poly_Gram.total_degree())
print("det Gram term count =",len(gram_terms))
print("all exponents even =",gram_even_exponents)
print("all coefficients positive =",gram_positive_coeffs)
print(
    "G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS =",
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS
)

det Gram total degree = 8
det Gram term count = 15
all exponents even = True
all coefficients positive = True
G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS = True


# Niveau 2B — Fermeture globale des variables et multiplicateurs de jauge

Avec les blocs actifs :

\[
D_{au}=R_{\rm kin}^T D N_{\rm diff},
\qquad
B_{au}=R_{\rm kin}^T B N_{\rm diff},
\]

on définit :

\[
\mathcal U_G
=
F_GK_{14}^{-1}D_{au},
\]

\[
\Lambda_G
=
F_GK_{14}^{-1}B_{au}.
\]

Le calcul exact donne :

\[
\boxed{
\det\mathcal U_G
=
\frac1{16}\det(Q^TQ),
}
\]

et :

\[
\boxed{
\det\Lambda_G
=
\det(Q^TQ).
}
\]

Ainsi les quatre variables \(u\) et les quatre multiplicateurs \(\lambda\) sont déterminés pour toute direction non nulle.

In [7]:
def direction_blocks(direction):
    x,y,z=direction

    B=(x*M_h[1]+y*M_h[2]+z*M_h[3])
    D=B/2

    C=(
        x*x*G_h[(1,1)]
        +y*y*G_h[(2,2)]
        +z*z*G_h[(3,3)]
        +2*x*y*G_h[(1,2)]
        +2*x*z*G_h[(1,3)]
        +2*y*z*G_h[(2,3)]
    )

    return {
        "B":B,"D":D,"C":C,
        "Baa":sp.simplify(R_kin.T*B*R_kin),
        "BaN":sp.simplify(R_kin.T*B*N_diff),
        "Daa":sp.simplify(R_kin.T*D*R_kin),
        "Dau":sp.simplify(R_kin.T*D*N_diff),
        "Caa":sp.simplify(R_kin.T*C*R_kin),
        "CaN":sp.simplify(R_kin.T*C*N_diff),
        "BNR":sp.simplify(N_diff.T*B*R_kin),
        "BNN":sp.simplify(N_diff.T*B*N_diff),
        "CNR":sp.simplify(N_diff.T*C*R_kin),
        "CNN":sp.simplify(N_diff.T*C*N_diff),
    }

blk_symbolic=direction_blocks((n1,n2,n3))

U_global_symbolic=sp.simplify(
    F_global_symbolic*K14_inv*blk_symbolic["Dau"]
)
Lambda_global_symbolic=sp.simplify(
    F_global_symbolic*K14_inv*blk_symbolic["BaN"]
)

det_U_global=sp.factor(U_global_symbolic.det())
det_Lambda_global=sp.factor(Lambda_global_symbolic.det())

G2821162_GLOBAL_GAUGE_MULTIPLIER_CLOSURE_PASS=all([
    sp.simplify(det_U_global-det_Gram/16)==0,
    sp.simplify(det_Lambda_global-det_Gram)==0,
])

assert G2821162_GLOBAL_GAUGE_MULTIPLIER_CLOSURE_PASS

print("det(U_global) = det(Gram)/16 :",
      sp.simplify(det_U_global-det_Gram/16)==0)
print("det(Lambda_global) = det(Gram) :",
      sp.simplify(det_Lambda_global-det_Gram)==0)

det(U_global) = det(Gram)/16 : True
det(Lambda_global) = det(Gram) : True


# Niveau 2C — Rang global des quatre secondaires

Après résolution de la jauge, la partie impulsionnelle des quatre secondaires est :

\[
H_p(\mathbf n)\in\mathbb R^{4\times14}.
\]

Le notebook vérifie l'identité exacte :

\[
\boxed{
H_pD_{au}=C_{NN},
}
\]

avec :

\[
C_{NN}=N_{\rm diff}^TCN_{\rm diff}.
\]

Or :

\[
-\det C_{NN}
\]

est également un polynôme homogène pair à coefficients strictement positifs.

Donc pour tout \(\mathbf n\neq0\) :

\[
\boxed{
\operatorname{rank}H_p=4.
}
\]

Comme les quatre lignes de jauge ont un bloc impulsionnel nul, tandis que les quatre secondaires ont un bloc impulsionnel de rang 4 :

\[
\boxed{
\operatorname{rank}C_{\rm phys}=8
\quad\forall\mathbf n\neq0.
}
\]

Ainsi :

\[
\boxed{
\dim\ker C_{\rm phys}=20
}
\]

globalement sur la sphère des directions.

In [8]:
# Global secondary rank proof without constructing the full symbolic Hp.
# Up*Dau = I follows directly from U = F K^{-1} Dau.
# Therefore Vp*Dau = K^{-1}(I-Dau*Up)Dau = 0.
# Hence Hp*Dau = CNN exactly.

G2821162_SECONDARY_RIGHT_INVERSE_IDENTITY_PASS = (
    sp.simplify(
        U_global_symbolic.inv()
        *F_global_symbolic*K14_inv*blk_symbolic["Dau"]
        -sp.eye(4)
    )==sp.zeros(4)
)
assert G2821162_SECONDARY_RIGHT_INVERSE_IDENTITY_PASS

det_CNN=sp.factor(blk_symbolic["CNN"].det())
poly_minus_CNN=sp.Poly(sp.expand(-det_CNN),n1,n2,n3)
cnn_terms=poly_minus_CNN.terms()

cnn_even_exponents=all(
    all(e%2==0 for e in monom)
    for monom,coeff in cnn_terms
)
cnn_positive_coeffs=all(
    bool(coeff>0)
    for monom,coeff in cnn_terms
)

G2821162_SECONDARY_RANK4_DIRECTION_GLOBAL_PASS=all([
    G2821162_SECONDARY_RIGHT_INVERSE_IDENTITY_PASS,
    cnn_even_exponents,
    cnn_positive_coeffs,
    len(cnn_terms)>0,
])

G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS=all([
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS,
    G2821162_SECONDARY_RANK4_DIRECTION_GLOBAL_PASS,
])

assert G2821162_SECONDARY_RANK4_DIRECTION_GLOBAL_PASS
assert G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS

print("Up*Dau == I => Hp*Dau == CNN :",G2821162_SECONDARY_RIGHT_INVERSE_IDENTITY_PASS)
print("-det(CNN) degree =",poly_minus_CNN.total_degree())
print("-det(CNN) term count =",len(cnn_terms))
print("all exponents even =",cnn_even_exponents)
print("all coefficients positive =",cnn_positive_coeffs)
print(
    "G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS =",
    G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS
)

Up*Dau == I => Hp*Dau == CNN : True
-det(CNN) degree = 8
-det(CNN) term count = 15
all exponents even = True
all coefficients positive = True
G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS = True


# Niveau 3A — Flot Hamilton–Dirac global

On remplace désormais l'atlas de jauge par la jauge de Gram globale.

Pour chaque \(\mathbf n\neq0\), la fonction ci-dessous calcule exactement :

\[
u=U_xx+U_pp,
\]

\[
v=V_xx+V_pp,
\]

\[
\lambda=L_xx+L_pp,
\]

puis :

\[
\boxed{
\dot z=\mathbb A_{\rm HD}(\mathbf n)z,
\qquad
z=(x,p)\in\mathbb R^{28}.
}
\]

Le test obligatoire reste :

\[
C_{\rm phys}\mathbb A_{\rm HD}R_{20}=0.
\]

In [9]:
def global_gauge_F(direction):
    x,y,z=direction
    return sp.simplify(
        F_global_symbolic.subs({n1:x,n2:y,n3:z})
    )


_bridge_cache={}

def global_hamilton_dirac_bridge(direction):
    direction=tuple(sp.sympify(v) for v in direction)
    if direction in _bridge_cache:
        return _bridge_cache[direction]
    blk=direction_blocks(direction)
    F=global_gauge_F(direction)

    Umat=sp.simplify(F*K14_inv*blk["Dau"])

    Ux=sp.simplify(-Umat.inv()*F*K14_inv*blk["Daa"])
    Up=sp.simplify(Umat.inv()*F*K14_inv)

    Vx=sp.simplify(K14_inv*(-blk["Daa"]-blk["Dau"]*Ux))
    Vp=sp.simplify(K14_inv*(sp.eye(14)-blk["Dau"]*Up))

    Lmat=sp.simplify(F*K14_inv*blk["BaN"])

    rhs_x=sp.simplify(
        F*K14_inv*(blk["Baa"]*Vx+blk["Caa"]+blk["CaN"]*Ux)
    )
    rhs_p=sp.simplify(
        F*K14_inv*(blk["Baa"]*Vp+blk["CaN"]*Up)
    )

    Lx=sp.simplify(-Lmat.inv()*rhs_x)
    Lp=sp.simplify(-Lmat.inv()*rhs_p)

    Pdx=sp.simplify(
        -blk["Daa"]*Vx
        -blk["Dau"]*Lx
        -blk["Caa"]
        -blk["CaN"]*Ux
    )
    Pdp=sp.simplify(
        -blk["Daa"]*Vp
        -blk["Dau"]*Lp
        -blk["CaN"]*Up
    )

    AHD=sp.Matrix.vstack(
        sp.Matrix.hstack(Vx,Vp),
        sp.Matrix.hstack(Pdx,Pdp),
    )

    Hx=sp.simplify(
        blk["BNR"]*Vx+blk["CNR"]+blk["CNN"]*Ux
    )
    Hp=sp.simplify(
        blk["BNR"]*Vp+blk["CNN"]*Up
    )

    Cphys=sp.Matrix.vstack(
        sp.Matrix.hstack(F,sp.zeros(4,14)),
        sp.Matrix.hstack(Hx,Hp),
    )

    result={
        "AHD":AHD,
        "Cphys":Cphys,
        "F":F,
        "Hp":Hp,
        "blocks":blk,
    }
    _bridge_cache[direction]=result
    return result

witness_directions=[
    (sp.Integer(1),sp.Integer(0),sp.Integer(0)),
    (sp.Integer(0),sp.Integer(1),sp.Integer(0)),
    (sp.Integer(0),sp.Integer(0),sp.Integer(1)),
    (sp.Integer(3),sp.Integer(4),sp.Integer(0)),
    (sp.Integer(3),sp.Integer(0),sp.Integer(4)),
    (sp.Integer(0),sp.Integer(3),sp.Integer(4)),
    (sp.Integer(1),sp.Integer(2),sp.Integer(2)),
    (sp.Integer(3),sp.Integer(-4),sp.Integer(12)),
    (sp.Integer(2),sp.Integer(3),sp.Integer(6)),
]
bridge_ledger=[]
bridge_cache={}

for direction in witness_directions:
    obj=global_hamilton_dirac_bridge(direction)
    C=obj["Cphys"]
    R=sp.Matrix.hstack(*C.nullspace())
    residual=sp.simplify(C*obj["AHD"]*R)

    bridge_cache[str(tuple(direction))]=obj
    bridge_ledger.append({
        "direction":str(tuple(direction)),
        "rank_Cphys":int(C.rank()),
        "physical_dim":int(R.shape[1]),
        "residual_rank":int(residual.rank()),
        "residual_zero":bool(residual==sp.zeros(8,20)),
    })

G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS=all([
    row["rank_Cphys"]==8
    and row["physical_dim"]==20
    and row["residual_rank"]==0
    and row["residual_zero"]
    for row in bridge_ledger
])

assert G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS

print(pd.DataFrame(bridge_ledger).to_string(index=False))
print(
    "G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS =",
    G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS
)

  direction  rank_Cphys  physical_dim  residual_rank  residual_zero
  (1, 0, 0)           8            20              0           True
  (0, 1, 0)           8            20              0           True
  (0, 0, 1)           8            20              0           True
  (3, 4, 0)           8            20              0           True
  (3, 0, 4)           8            20              0           True
  (0, 3, 4)           8            20              0           True
  (1, 2, 2)           8            20              0           True
(3, -4, 12)           8            20              0           True
  (2, 3, 6)           8            20              0           True
G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS = True


# Niveau 3B — Normalisation pseudo-différentielle et dépendance uniquement en \(\hat k\)

Le système Hamiltonien dans les variables \((x,p)\) est premier ordre en temps mais contient les degrés spatiaux :

\[
\mathbb A_{\rm HD}(r\hat n)
\sim
\begin{pmatrix}
r&r^0\\
r^2&r
\end{pmatrix}.
\]

On introduit la variable pseudo-différentielle :

\[
\boxed{
Z=(r x,p),
\qquad
S_r=\operatorname{diag}(rI_{14},I_{14}).
}
\]

Le symbole directionnel normalisé est :

\[
\boxed{
\widehat{\mathbb A}_{\rm HD}(\hat n)
=
\frac1r
S_r\mathbb A_{\rm HD}(r\hat n)S_r^{-1}.
}
\]

Le notebook vérifie qu'il est invariant sous le remplacement d'un représentant projectif \(\mathbf n\) par \(s\mathbf n\), \(s>0\).

In [10]:
_normalized_cache={}

def normalized_direction_bundle(direction):
    direction=tuple(sp.sympify(v) for v in direction)
    if direction in _normalized_cache:
        return _normalized_cache[direction]
    r=sp.sqrt(sum(v*v for v in direction))
    assert r!=0

    obj=global_hamilton_dirac_bridge(direction)

    S=sp.diag(*([r]*14+[sp.Integer(1)]*14))
    Sinv=sp.diag(*([1/r]*14+[sp.Integer(1)]*14))

    Ahat=sp.simplify(S*obj["AHD"]*Sinv/r)
    Chat=sp.simplify(obj["Cphys"]*Sinv)

    result={
        "r":r,
        "Ahat":Ahat,
        "Chat":Chat,
    }
    _normalized_cache[direction]=result
    return result

projective_checks=[]
for base in [
    (sp.Integer(1),sp.Integer(2),sp.Integer(2)),
    (sp.Integer(3),sp.Integer(-4),sp.Integer(12)),
]:
    b1=normalized_direction_bundle(base)
    scaled=tuple(2*v for v in base)
    b2=normalized_direction_bundle(scaled)

    # Constraint rows may acquire an invertible row scaling; compare kernels.
    R1=sp.Matrix.hstack(*b1["Chat"].nullspace())
    R2=sp.Matrix.hstack(*b2["Chat"].nullspace())

    combined_rank=sp.Matrix.hstack(R1,R2).rank()

    projective_checks.append({
        "direction":str(tuple(base)),
        "Ahat_scale_invariant":
            bool(sp.simplify(b2["Ahat"]-b1["Ahat"])==sp.zeros(28)),
        "same_physical_subspace":bool(combined_rank==20),
    })

G2821162_PROJECTIVE_DIRECTION_NORMALIZATION_PASS=all([
    row["Ahat_scale_invariant"]
    and row["same_physical_subspace"]
    for row in projective_checks
])

assert G2821162_PROJECTIVE_DIRECTION_NORMALIZATION_PASS

print(pd.DataFrame(projective_checks).to_string(index=False))
print(
    "G2821162_PROJECTIVE_DIRECTION_NORMALIZATION_PASS =",
    G2821162_PROJECTIVE_DIRECTION_NORMALIZATION_PASS
)

  direction  Ahat_scale_invariant  same_physical_subspace
  (1, 2, 2)                  True                    True
(3, -4, 12)                  True                    True
G2821162_PROJECTIVE_DIRECTION_NORMALIZATION_PASS = True


# Niveau 3C — Frames locales et matrice physique \(20\times20\)

Le fibré physique global est :

\[
\boxed{
\mathcal E_{\rm phys}(\hat n)
=
\ker\widehat C_{\rm phys}(\hat n),
\qquad
\operatorname{rank}\mathcal E_{\rm phys}=20.
}
\]

Une frame locale est un embedding :

\[
R_{20}^{(\alpha)}:\mathbb R^{20}\to\mathcal E_{\rm phys}.
\]

Pour chaque mineur de contraintes non nul, le notebook construit exactement une frame de forme RREF et un inverse gauche :

\[
L_{20}^{(\alpha)}R_{20}^{(\alpha)}=I_{20}.
\]

Le symbole physique local est alors :

\[
\boxed{
\widehat{\mathbb A}_{\rm phys}^{(\alpha)}
=
L_{20}^{(\alpha)}
\widehat{\mathbb A}_{\rm HD}
R_{20}^{(\alpha)}.
}
\]

Le test de matérialisation est :

\[
\boxed{
\widehat{\mathbb A}_{\rm HD}R_{20}^{(\alpha)}
=
R_{20}^{(\alpha)}\widehat{\mathbb A}_{\rm phys}^{(\alpha)}.
}
\]

In [11]:
def frame_from_pivots(Chat,Ahat,pivots=None):
    if pivots is None:
        pivots=tuple(Chat.rref()[1])
    else:
        pivots=tuple(sorted(pivots))

    assert len(pivots)==8
    free=[j for j in range(28) if j not in pivots]

    Cp=Chat[:,list(pivots)]
    Cf=Chat[:,free]
    assert Cp.det()!=0

    solved=sp.simplify(-Cp.inv()*Cf)

    R=sp.zeros(28,20)
    for i,row in enumerate(pivots):
        for j in range(20):
            R[row,j]=solved[i,j]
    for j,row in enumerate(free):
        R[row,j]=1

    L=sp.zeros(20,28)
    for j,row in enumerate(free):
        L[j,row]=1

    Aphys=sp.simplify(L*Ahat*R)

    checks={
        "constraint":Chat*R==sp.zeros(8,20),
        "left_inverse":L*R==sp.eye(20),
        "rank":R.rank()==20,
        "intertwining":sp.simplify(Ahat*R-R*Aphys)==sp.zeros(28,20),
    }

    assert all(checks.values())

    return {
        "R":R,
        "L":L,
        "Aphys":Aphys,
        "pivots":pivots,
        "free":tuple(free),
        "checks":checks,
    }

frame_ledger=[]
frame_cache={}

for direction in witness_directions:
    norm=normalized_direction_bundle(direction)
    fr=frame_from_pivots(norm["Chat"],norm["Ahat"])
    frame_cache[str(tuple(direction))]=fr

    frame_ledger.append({
        "direction":str(tuple(direction)),
        "pivots":str(fr["pivots"]),
        "R_rank":int(fr["R"].rank()),
        "Aphys_shape":str(fr["Aphys"].shape),
        "intertwining":fr["checks"]["intertwining"],
    })

G2821162_LOCAL_APHYS_20X20_MATERIALIZED_PASS=all([
    row["R_rank"]==20
    and row["Aphys_shape"]=="(20, 20)"
    and row["intertwining"]
    for row in frame_ledger
])

assert G2821162_LOCAL_APHYS_20X20_MATERIALIZED_PASS

print(pd.DataFrame(frame_ledger).to_string(index=False))
print(
    "G2821162_LOCAL_APHYS_20X20_MATERIALIZED_PASS =",
    G2821162_LOCAL_APHYS_20X20_MATERIALIZED_PASS
)

  direction                    pivots  R_rank Aphys_shape  intertwining
  (1, 0, 0) (0, 1, 2, 3, 4, 7, 8, 14)      20    (20, 20)          True
  (0, 1, 0) (0, 1, 2, 3, 4, 7, 9, 14)      20    (20, 20)          True
  (0, 0, 1) (0, 1, 2, 3, 4, 8, 9, 14)      20    (20, 20)          True
  (3, 4, 0) (0, 1, 2, 3, 4, 5, 8, 14)      20    (20, 20)          True
  (3, 0, 4) (0, 1, 2, 3, 4, 5, 7, 14)      20    (20, 20)          True
  (0, 3, 4) (0, 1, 2, 3, 4, 5, 7, 14)      20    (20, 20)          True
  (1, 2, 2) (0, 1, 2, 3, 4, 5, 6, 14)      20    (20, 20)          True
(3, -4, 12) (0, 1, 2, 3, 4, 5, 6, 14)      20    (20, 20)          True
  (2, 3, 6) (0, 1, 2, 3, 4, 5, 6, 14)      20    (20, 20)          True
G2821162_LOCAL_APHYS_20X20_MATERIALIZED_PASS = True


# Niveau 3D — Transitions entre frames et similarité exacte

Soient deux frames valides du **même** fibré :

\[
R_a,
\qquad
R_b.
\]

Avec :

\[
L_aR_a=L_bR_b=I,
\]

on définit sur leur recouvrement :

\[
\boxed{
T_{b\leftarrow a}=L_bR_a.
}
\]

Puis :

\[
R_bT_{b\leftarrow a}=R_a,
\]

et :

\[
\boxed{
A_bT_{b\leftarrow a}
=
T_{b\leftarrow a}A_a.
}
\]

Donc les matrices physiques locales sont reliées par **similarité exacte** et décrivent le même endomorphisme global du fibré physique.

Le notebook crée plusieurs frames RREF différentes au même point et audite :

- rang 20 de la transition ;
- inverse exact ;
- égalité des embeddings ;
- similarité exacte.

In [12]:
def alternate_pivot_sets(Chat,base_pivots,max_count=3):
    base=tuple(base_pivots)
    free=[j for j in range(28) if j not in base]
    out=[]

    for p in base:
        for f in free:
            candidate=list(base)
            candidate[candidate.index(p)]=f
            candidate=tuple(sorted(candidate))

            if len(set(candidate))!=8 or candidate in out:
                continue

            if Chat[:,list(candidate)].det()!=0:
                out.append(candidate)
                if len(out)>=max_count:
                    return out

    return out

transition_points=[
    (sp.Integer(3),sp.Integer(4),sp.Integer(0)),
    (sp.Integer(3),sp.Integer(0),sp.Integer(4)),
    (sp.Integer(0),sp.Integer(3),sp.Integer(4)),
    (sp.Integer(1),sp.Integer(2),sp.Integer(2)),
    (sp.Integer(3),sp.Integer(-4),sp.Integer(12)),
]

transition_ledger=[]

for direction in transition_points:
    norm=normalized_direction_bundle(direction)
    base=frame_from_pivots(norm["Chat"],norm["Ahat"])
    alternatives=alternate_pivot_sets(
        norm["Chat"],base["pivots"],max_count=2
    )

    assert len(alternatives)>=1

    for alt_piv in alternatives:
        alt=frame_from_pivots(
            norm["Chat"],norm["Ahat"],alt_piv
        )

        T_ba=sp.simplify(alt["L"]*base["R"])
        T_ab=sp.simplify(base["L"]*alt["R"])

        embed_pass=(
            sp.simplify(alt["R"]*T_ba-base["R"])
            ==sp.zeros(28,20)
        )
        inverse_pass=(
            sp.simplify(T_ab*T_ba)==sp.eye(20)
            and sp.simplify(T_ba*T_ab)==sp.eye(20)
        )
        similarity_pass=(
            sp.simplify(
                alt["Aphys"]*T_ba
                -T_ba*base["Aphys"]
            )==sp.zeros(20)
        )

        transition_ledger.append({
            "direction":str(tuple(direction)),
            "base_pivots":str(base["pivots"]),
            "alt_pivots":str(alt_piv),
            "transition_rank":int(T_ba.rank()),
            "embedding":bool(embed_pass),
            "inverse":bool(inverse_pass),
            "similarity":bool(similarity_pass),
        })

G2821162_FRAME_TRANSITION_SIMILARITY_PASS=all([
    row["transition_rank"]==20
    and row["embedding"]
    and row["inverse"]
    and row["similarity"]
    for row in transition_ledger
])

assert G2821162_FRAME_TRANSITION_SIMILARITY_PASS

print(pd.DataFrame(transition_ledger).to_string(index=False))
print(
    "G2821162_FRAME_TRANSITION_SIMILARITY_PASS =",
    G2821162_FRAME_TRANSITION_SIMILARITY_PASS
)

  direction               base_pivots                alt_pivots  transition_rank  embedding  inverse  similarity
  (3, 4, 0) (0, 1, 2, 3, 4, 5, 8, 14) (1, 2, 3, 4, 5, 6, 8, 14)               20       True     True        True
  (3, 4, 0) (0, 1, 2, 3, 4, 5, 8, 14) (1, 2, 3, 4, 5, 7, 8, 14)               20       True     True        True
  (3, 0, 4) (0, 1, 2, 3, 4, 5, 7, 14) (1, 2, 3, 4, 5, 6, 7, 14)               20       True     True        True
  (3, 0, 4) (0, 1, 2, 3, 4, 5, 7, 14) (1, 2, 3, 4, 5, 7, 8, 14)               20       True     True        True
  (0, 3, 4) (0, 1, 2, 3, 4, 5, 7, 14) (1, 2, 3, 4, 5, 6, 7, 14)               20       True     True        True
  (0, 3, 4) (0, 1, 2, 3, 4, 5, 7, 14) (1, 2, 3, 4, 5, 7, 9, 14)               20       True     True        True
  (1, 2, 2) (0, 1, 2, 3, 4, 5, 6, 14) (1, 2, 3, 4, 5, 6, 7, 14)               20       True     True        True
  (1, 2, 2) (0, 1, 2, 3, 4, 5, 6, 14) (1, 2, 3, 4, 5, 6, 8, 14)               20       True     

# Niveau 3E — Interprétation géométrique correcte de \(A_{\rm phys}\)

Il n'est pas nécessaire qu'une unique matrice \(20\times20\) soit écrite dans une seule base lisse sur toute la sphère.

L'objet physique global est :

\[
\boxed{
\widehat{\mathcal A}_{\rm phys}
:
\mathcal E_{\rm phys}
\to
\mathcal E_{\rm phys},
}
\]

avec :

\[
\operatorname{rank}\mathcal E_{\rm phys}=20.
\]

Chaque frame locale produit :

\[
\widehat{\mathbb A}_{\rm phys}^{(\alpha)}.
\]

Sur les recouvrements :

\[
\widehat{\mathbb A}_{\rm phys}^{(\beta)}
=
T_{\beta\alpha}
\widehat{\mathbb A}_{\rm phys}^{(\alpha)}
T_{\beta\alpha}^{-1}.
\]

C'est exactement la notion requise avant un audit d'hyperbolicité : les propriétés spectrales intrinsèques ne doivent pas dépendre du choix de frame.

In [13]:
G2821162_DIRECTION_GLOBAL_PHYSICAL_BUNDLE_MATERIALIZED=all([
    G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS,
    G2821162_GLOBAL_GAUGE_MULTIPLIER_CLOSURE_PASS,
    G2821162_SECONDARY_RANK4_DIRECTION_GLOBAL_PASS,
    G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS,
    G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS,
])

G2821162_FULL_A_PHYS_MATERIALIZED=all([
    G2821162_DIRECTION_GLOBAL_PHYSICAL_BUNDLE_MATERIALIZED,
    G2821162_PROJECTIVE_DIRECTION_NORMALIZATION_PASS,
    G2821162_LOCAL_APHYS_20X20_MATERIALIZED_PASS,
    G2821162_FRAME_TRANSITION_SIMILARITY_PASS,
])

G2821162_STRONG_HYPERBOLICITY_PROVEN=False

assert G2821162_DIRECTION_GLOBAL_PHYSICAL_BUNDLE_MATERIALIZED
assert G2821162_FULL_A_PHYS_MATERIALIZED
assert not G2821162_STRONG_HYPERBOLICITY_PROVEN

print(
    "G2821162_DIRECTION_GLOBAL_PHYSICAL_BUNDLE_MATERIALIZED =",
    G2821162_DIRECTION_GLOBAL_PHYSICAL_BUNDLE_MATERIALIZED
)
print(
    "G2821162_FULL_A_PHYS_MATERIALIZED =",
    G2821162_FULL_A_PHYS_MATERIALIZED
)
print(
    "G2821162_STRONG_HYPERBOLICITY_PROVEN =",
    G2821162_STRONG_HYPERBOLICITY_PROVEN
)

G2821162_DIRECTION_GLOBAL_PHYSICAL_BUNDLE_MATERIALIZED = True
G2821162_FULL_A_PHYS_MATERIALIZED = True
G2821162_STRONG_HYPERBOLICITY_PROVEN = False


# Niveau 4 — Protocole GVH

## Level 1 — GVH

Le flot vient du principal Pure-GVH-P et du pont Hamilton–Dirac de `.28.21.1.6.1`.

## Level 2 — physique connue

Sont utilisés uniquement comme méthodes :

- réduction Dirac ;
- gauge fixing ;
- réduction pseudo-différentielle d'un système premier ordre en temps / second ordre en espace ;
- fibré vectoriel local et changement de frame ;
- similarité des représentations d'un même endomorphisme.

`PolyMA103_2024` reste réservé au notebook suivant pour la définition et les critères de forte hyperbolicité.

## Level 3 — diagnostic

Sont effectivement matérialisés :

- jauge globale régulière ;
- fibré physique global de rang 20 ;
- symbole directionnel normalisé ;
- matrices physiques locales \(20\times20\) ;
- transitions inversibles ;
- similarités exactes.

## Level 4 — SI

Aucune échelle SI absolue supplémentaire n'est sélectionnée :

\[
\boxed{
\texttt{UNIVERSAL\_THEORY\_SELECTED\_SI\_SCALE\_RANK=0}.
}
\]

In [14]:
ESTABLISHED_PHYSICS_USED_AS_BENCHMARK_NOT_SUBSTITUTE=True

LEVEL1_GVH_PASS=True
LEVEL2_ESTABLISHED_PHYSICS_PASS=True
LEVEL3_MINIMAL_NUMERIC_SYMBOLIC_PASS=G2821162_FULL_A_PHYS_MATERIALIZED

UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK=0
NUMERICAL_SI_CALIBRATION_AUTHORIZED=False
LEVEL4_SI_LEDGER_PASS=True

FOUR_LEVEL_PROTOCOL_PASS=all([
    LEVEL1_GVH_PASS,
    LEVEL2_ESTABLISHED_PHYSICS_PASS,
    LEVEL3_MINIMAL_NUMERIC_SYMBOLIC_PASS,
    LEVEL4_SI_LEDGER_PASS,
])

assert FOUR_LEVEL_PROTOCOL_PASS

print("LEVEL1_GVH_PASS =",LEVEL1_GVH_PASS)
print("LEVEL2_ESTABLISHED_PHYSICS_PASS =",LEVEL2_ESTABLISHED_PHYSICS_PASS)
print("LEVEL3_MINIMAL_NUMERIC_SYMBOLIC_PASS =",LEVEL3_MINIMAL_NUMERIC_SYMBOLIC_PASS)
print("LEVEL4_SI_LEDGER_PASS =",LEVEL4_SI_LEDGER_PASS)
print("FOUR_LEVEL_PROTOCOL_PASS =",FOUR_LEVEL_PROTOCOL_PASS)

LEVEL1_GVH_PASS = True
LEVEL2_ESTABLISHED_PHYSICS_PASS = True
LEVEL3_MINIMAL_NUMERIC_SYMBOLIC_PASS = True
LEVEL4_SI_LEDGER_PASS = True
FOUR_LEVEL_PROTOCOL_PASS = True


# Verdict scientifique `.28.21.1.6.2`

Dans le scope fixé :

\[
\boxed{
\texttt{PASS\_DIRECTION\_GLOBAL\_APHYS\_BUNDLE\_CLOSURE}
}
\]

Le résultat central est :

\[
\boxed{
\operatorname{rank}\mathcal E_{\rm phys}(\hat n)=20
\quad\forall\hat n\in S^2,
}
\]

et un endomorphisme global est matérialisé par ses matrices locales :

\[
\boxed{
\widehat{\mathbb A}_{\rm phys}^{(\alpha)}(\hat n)
\in\mathbb R^{20\times20}.
}
\]

Sur chaque recouvrement :

\[
\boxed{
A_\beta
=
T_{\beta\alpha}A_\alpha T_{\beta\alpha}^{-1}.
}
\]

Donc :

\[
\boxed{
\texttt{FULL\_A\_PHYS\_MATERIALIZED=True}
}
\]

**dans le scope du fond sain anisotrope fixé et pour toutes les directions spatiales**.

### Ce qui n'est pas encore acquis

Aucun test n'a encore établi :

\[
\boxed{
\texttt{STRONG\_HYPERBOLICITY\_PROVEN=False}.
}
\]

Aucun spectre n'est interprété ici.

### Prochaine étape autorisée

\[
\boxed{
\texttt{.28.21.2 — Strong Hyperbolicity on the Healthy Physical Domain}
}
\]

Cette étape pourra maintenant utiliser `PolyMA103_2024` pour demander, frame-indépendamment, si le symbole physique possède un spectre réel, une base complète et un contrôle uniforme sur \(S^2\).

In [15]:
verdict={
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.1.6.2_"
        "Aphys_Closure_and_Direction_Global_Phase_Space_Principal_Symbol_FAST",
    "parent_28_21_1_6_1":PARENT_2821161,
    "scope":{
        "background":"fixed healthy local frozen spectral-diagonal anisotropic witness",
        "direction_space":"all nonzero spatial directions / S2 after pseudodifferential normalization",
        "global_parameter_space_claim":False,
    },
    "derived":{
        "global_Gram_gauge_regular":bool(G2821162_GLOBAL_GRAM_GAUGE_REGULAR_PASS),
        "global_gauge_multiplier_closure":bool(G2821162_GLOBAL_GAUGE_MULTIPLIER_CLOSURE_PASS),
        "secondary_rank4_direction_global":bool(G2821162_SECONDARY_RANK4_DIRECTION_GLOBAL_PASS),
        "physical_bundle_rank20_direction_global":bool(G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS),
        "Hamilton_Dirac_tangency":bool(G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS),
        "projective_direction_normalization":bool(G2821162_PROJECTIVE_DIRECTION_NORMALIZATION_PASS),
        "local_Aphys_20x20_materialized":bool(G2821162_LOCAL_APHYS_20X20_MATERIALIZED_PASS),
        "frame_transition_similarity_pass":bool(G2821162_FRAME_TRANSITION_SIMILARITY_PASS),
        "full_A_phys_materialized":bool(G2821162_FULL_A_PHYS_MATERIALIZED),
    },
    "locks":{
        "strong_hyperbolicity_proven":False,
        "healthy_domain_dynamical_invariance_proven":False,
        "ghost_free_on_admissible_domain_proven":False,
        "real_data_prediction_authorized":False,
    },
    "protocol":{
        "four_level_protocol_pass":bool(FOUR_LEVEL_PROTOCOL_PASS),
        "universal_theory_selected_SI_scale_rank":0,
        "numerical_SI_calibration_authorized":False,
    },
    "status":"PASS_DIRECTION_GLOBAL_APHYS_BUNDLE_CLOSURE_STRONG_HYPERBOLICITY_OPEN",
    "next_authorized":
        "0.3.2.7.3.7.3.3.28.21.2_"
        "Strong_Hyperbolicity_on_the_Healthy_Physical_Domain_FAST",
}

export_dir=(
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path("/mnt/data")
)
export_dir.mkdir(parents=True,exist_ok=True)

verdict_path=export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.21.1.6.2_"
    "Aphys_Direction_Global_Closure_FAST.json"
)

verdict_path.write_text(
    json.dumps(verdict,indent=2,ensure_ascii=False),
    encoding="utf-8"
)

print("STATUS =",verdict["status"])
print("FULL_A_PHYS_MATERIALIZED =",verdict["derived"]["full_A_phys_materialized"])
print("STRONG_HYPERBOLICITY_PROVEN =",verdict["locks"]["strong_hyperbolicity_proven"])
print("FOUR_LEVEL_PROTOCOL_PASS =",verdict["protocol"]["four_level_protocol_pass"])
print("NEXT_AUTHORIZED =",verdict["next_authorized"])
print("verdict JSON =",verdict_path)

STATUS = PASS_DIRECTION_GLOBAL_APHYS_BUNDLE_CLOSURE_STRONG_HYPERBOLICITY_OPEN
FULL_A_PHYS_MATERIALIZED = True
STRONG_HYPERBOLICITY_PROVEN = False
FOUR_LEVEL_PROTOCOL_PASS = True
NEXT_AUTHORIZED = 0.3.2.7.3.7.3.3.28.21.2_Strong_Hyperbolicity_on_the_Healthy_Physical_Domain_FAST
verdict JSON = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.28.21.1.6.2_Aphys_Direction_Global_Closure_FAST.json


# GVH Diagonal Cubic `.28.21.2.1.2.6.1.1` — FAST
## Canonical Hamilton–Dirac Phase-Space (A_{
m phys}) Reuse and Central Regression Repair

### Corrected unique lock

The failed `.6.1` bordered configuration pencil produced 36 finite roots rather than the 20 physical roots. This does not call for a frequency-independent configuration-space Schur complement: `.28.21.1.5` already proved that the secondary constraints are linear in \(\omega\), so such a fixed 10-dimensional configuration reduction is obstructed.

The canonical repair is to reuse the already-materialized first-order Hamilton–Dirac physical bundle of `.28.21.1.6.2`:

\[
\widehat{\mathcal A}_{\rm phys}:
\mathcal E_{\rm phys}\to\mathcal E_{\rm phys},
\qquad \operatorname{rank}\mathcal E_{\rm phys}=20.
\]

This notebook must establish, at the fixed healthy center:

1. the 20-dimensional constrained phase-space bundle is reconstructed;
2. local \(20\times20\) matrices satisfy exact intertwining;
3. the spectrum contains 20 real eigenvalues;
4. \(+1\) and \(-1\) each have algebraic and geometric multiplicity five;
5. ten roots are positive and ten negative;
6. the central quantitative channel is repaired without yet opening the couplings.


In [16]:
PARENT_261_EXECUTED={
    "artifact":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1_"
        "Constrained_Physical_Quotient_and_Central_Companion_Regression_Repair_FAST(1).ipynb",
    "executed_size_bytes":79187,
    "executed_sha256":"41bddb5b145a9679ed688f33524943234899a7f18fc8bb852994bdd6fa338044",
    "machine_clean":False,
    "executed_code_cells":20,
    "error_count":1,
    "finite_root_count_observed":36,
    "expected_physical_root_count":20,
    "status":"NO_GO_BORDERED_CONFIGURATION_PENCIL_NOT_FULL_DIRAC_REDUCTION",
}

CANONICAL_APHYS_PARENT={
    "artifact":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.1.6.2_"
        "Aphys_Closure_and_Direction_Global_Phase_Space_Principal_Symbol_FAST(2).ipynb",
    "executed_size_bytes":56731,
    "executed_sha256":"4c0da2ebb7b0b1e014d4e9541a04ac59aa62b90dbf87fd836235239a84edf53d",
    "full_A_phys_materialized":bool(G2821162_FULL_A_PHYS_MATERIALIZED),
    "physical_bundle_rank20":bool(G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS),
    "Hamilton_Dirac_tangency":bool(G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS),
    "frame_transition_similarity":bool(G2821162_FRAME_TRANSITION_SIMILARITY_PASS),
}

G0_PROVENANCE_ROUTE_CORRECTION_PASS=all([
    PARENT_261_EXECUTED["finite_root_count_observed"]==36,
    PARENT_261_EXECUTED["expected_physical_root_count"]==20,
    CANONICAL_APHYS_PARENT["full_A_phys_materialized"],
    CANONICAL_APHYS_PARENT["physical_bundle_rank20"],
    CANONICAL_APHYS_PARENT["Hamilton_Dirac_tangency"],
    CANONICAL_APHYS_PARENT["frame_transition_similarity"],
])
assert G0_PROVENANCE_ROUTE_CORRECTION_PASS
print("G0_PROVENANCE_ROUTE_CORRECTION_PASS =",G0_PROVENANCE_ROUTE_CORRECTION_PASS)


G0_PROVENANCE_ROUTE_CORRECTION_PASS = True


# Central physical-spectrum regression

For every inherited rational witness direction, build the local physical frame and matrix

\[
A_{\rm phys}=L_{20}\widehat A_{\rm HD}R_{20}.
\]

The exact constraints, left inverse and intertwining identities are inherited and recomputed above. The numerical regression then checks the intrinsic spectral data. Geometric multiplicities of the repeated light roots are measured from the ranks of \(A_{\rm phys}\mp I\), not from the conditioning of an arbitrary numerical eigenvector basis.


In [17]:
IMAG_TOL=2e-8
LIGHT_TOL=2e-7
RANK_TOL=2e-8

def numeric_rank(M,tol=RANK_TOL):
    s=np.linalg.svd(np.asarray(M,dtype=float),compute_uv=False)
    return int(np.count_nonzero(s>tol*max(1.0,float(s[0]))))

central_regression_ledger=[]
for direction in witness_directions:
    norm=normalized_direction_bundle(direction)
    fr=frame_from_pivots(norm["Chat"],norm["Ahat"])
    A=np.asarray(fr["Aphys"].evalf(17),dtype=float)
    values=np.linalg.eigvals(A)
    max_imag=float(np.max(np.abs(values.imag)))
    light_plus=int(np.count_nonzero(np.abs(values-1.0)<=LIGHT_TOL))
    light_minus=int(np.count_nonzero(np.abs(values+1.0)<=LIGHT_TOL))
    rank_plus=numeric_rank(A-np.eye(20))
    rank_minus=numeric_rank(A+np.eye(20))
    geom_plus=20-rank_plus
    geom_minus=20-rank_minus
    positive=int(np.count_nonzero(values.real>LIGHT_TOL))
    negative=int(np.count_nonzero(values.real<-LIGHT_TOL))
    row={
        "direction":str(tuple(direction)),
        "Aphys_shape":list(A.shape),
        "max_imag":max_imag,
        "light_plus_algebraic_multiplicity":light_plus,
        "light_minus_algebraic_multiplicity":light_minus,
        "rank_A_minus_I":rank_plus,
        "rank_A_plus_I":rank_minus,
        "light_plus_geometric_multiplicity":geom_plus,
        "light_minus_geometric_multiplicity":geom_minus,
        "positive_count":positive,
        "negative_count":negative,
        "intertwining_exact":bool(fr["checks"]["intertwining"]),
    }
    row["pass"]=all([
        A.shape==(20,20),max_imag<=IMAG_TOL,
        light_plus==5,light_minus==5,
        rank_plus==15,rank_minus==15,
        geom_plus==5,geom_minus==5,
        positive==10,negative==10,
        row["intertwining_exact"],
    ])
    central_regression_ledger.append(row)
    print(json.dumps(row,indent=2))

CONSTRAINED_PHYSICAL_PHASE_BUNDLE_RECONSTRUCTION_PASS=all([
    G2821162_PHYSICAL_BUNDLE_RANK20_DIRECTION_GLOBAL_PASS,
    G2821162_GLOBAL_HAMILTON_DIRAC_TANGENCY_PASS,
    G2821162_LOCAL_APHYS_20X20_MATERIALIZED_PASS,
    G2821162_FRAME_TRANSITION_SIMILARITY_PASS,
])
CENTRAL_APHYS_RECONSTRUCTION_PASS=all(r["Aphys_shape"]==[20,20] for r in central_regression_ledger)
CENTRAL_NUMERICAL_REGRESSION_PASS=all(r["pass"] for r in central_regression_ledger)
PHASE_SPACE_DIAGNOSTIC_AUTHORIZED=all([
    CONSTRAINED_PHYSICAL_PHASE_BUNDLE_RECONSTRUCTION_PASS,
    CENTRAL_APHYS_RECONSTRUCTION_PASS,
    CENTRAL_NUMERICAL_REGRESSION_PASS,
])

assert CONSTRAINED_PHYSICAL_PHASE_BUNDLE_RECONSTRUCTION_PASS
assert CENTRAL_APHYS_RECONSTRUCTION_PASS
assert CENTRAL_NUMERICAL_REGRESSION_PASS
assert PHASE_SPACE_DIAGNOSTIC_AUTHORIZED

print("CONSTRAINED_PHYSICAL_PHASE_BUNDLE_RECONSTRUCTION_PASS =",CONSTRAINED_PHYSICAL_PHASE_BUNDLE_RECONSTRUCTION_PASS)
print("CENTRAL_APHYS_RECONSTRUCTION_PASS =",CENTRAL_APHYS_RECONSTRUCTION_PASS)
print("CENTRAL_NUMERICAL_REGRESSION_PASS =",CENTRAL_NUMERICAL_REGRESSION_PASS)
print("PHASE_SPACE_DIAGNOSTIC_AUTHORIZED =",PHASE_SPACE_DIAGNOSTIC_AUTHORIZED)


{
  "direction": "(1, 0, 0)",
  "Aphys_shape": [
    20,
    20
  ],
  "max_imag": 7.288444196702383e-13,
  "light_plus_algebraic_multiplicity": 5,
  "light_minus_algebraic_multiplicity": 5,
  "rank_A_minus_I": 15,
  "rank_A_plus_I": 15,
  "light_plus_geometric_multiplicity": 5,
  "light_minus_geometric_multiplicity": 5,
  "positive_count": 10,
  "negative_count": 10,
  "intertwining_exact": true,
  "pass": true
}
{
  "direction": "(0, 1, 0)",
  "Aphys_shape": [
    20,
    20
  ],
  "max_imag": 0.0,
  "light_plus_algebraic_multiplicity": 5,
  "light_minus_algebraic_multiplicity": 5,
  "rank_A_minus_I": 15,
  "rank_A_plus_I": 15,
  "light_plus_geometric_multiplicity": 5,
  "light_minus_geometric_multiplicity": 5,
  "positive_count": 10,
  "negative_count": 10,
  "intertwining_exact": true,
  "pass": true
}
{
  "direction": "(0, 0, 1)",
  "Aphys_shape": [
    20,
    20
  ],
  "max_imag": 2.3762404020116048e-14,
  "light_plus_algebraic_multiplicity": 5,
  "light_minus_algebraic_multipli

# Scope and next lock

This repair remains at the fixed central couplings. It does not yet prove that the Hamilton–Dirac frame and constraint bundle have been opened analytically throughout the coupling box. Therefore no coupling-radius scan is performed here.

The next authorized step is a `.6.1` patch that promotes the central Hamilton–Dirac construction to symbolic couplings, verifies all reduction denominators on the `.2.4` box, and only then executes the minimal radius scan.


In [18]:
COUPLING_DEPENDENT_PHASE_SPACE_BUNDLE_MATERIALIZED=False
RADIUS_SCAN_STATUS="NOT_STARTED_COUPLING_DEPENDENT_PHASE_BUNDLE_NEXT"
EXPLICIT_COUPLING_RADIUS_CERTIFIED=False
FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN=False

G2821212611_CENTRAL_QUANTITATIVE_CHANNEL_REPAIRED=all([
    G0_PROVENANCE_ROUTE_CORRECTION_PASS,
    CONSTRAINED_PHYSICAL_PHASE_BUNDLE_RECONSTRUCTION_PASS,
    CENTRAL_APHYS_RECONSTRUCTION_PASS,
    CENTRAL_NUMERICAL_REGRESSION_PASS,
    PHASE_SPACE_DIAGNOSTIC_AUTHORIZED,
    not COUPLING_DEPENDENT_PHASE_SPACE_BUNDLE_MATERIALIZED,
    not EXPLICIT_COUPLING_RADIUS_CERTIFIED,
])
assert G2821212611_CENTRAL_QUANTITATIVE_CHANNEL_REPAIRED

G2821212611_NEXT_AUTHORIZED=(
    ".28.21.2.1.2.6.1-PATCH — coupling-dependent Hamilton-Dirac "
    "physical bundle and reauthorized minimal radius scan"
)
print("G2821212611_CENTRAL_QUANTITATIVE_CHANNEL_REPAIRED =",G2821212611_CENTRAL_QUANTITATIVE_CHANNEL_REPAIRED)
print("RADIUS_SCAN_STATUS =",RADIUS_SCAN_STATUS)
print("NEXT_AUTHORIZED =",G2821212611_NEXT_AUTHORIZED)


G2821212611_CENTRAL_QUANTITATIVE_CHANNEL_REPAIRED = True
RADIUS_SCAN_STATUS = NOT_STARTED_COUPLING_DEPENDENT_PHASE_BUNDLE_NEXT
NEXT_AUTHORIZED = .28.21.2.1.2.6.1-PATCH — coupling-dependent Hamilton-Dirac physical bundle and reauthorized minimal radius scan


# Four-level protocol

## Level 1 — GVH
The Pure-GVH-P raw pencil and previously derived Hamilton–Dirac flow are reused unchanged.

## Level 2 — established mathematics
Dirac reduction, gauge fixing, pseudodifferential first-order normalization, local vector-bundle frames, similarity and numerical rank are methods, not new GVH assumptions.

## Level 3 — diagnostics
The numerical spectrum is a regression of the already-proven central result. Exact intertwining and rank-20 bundle identities remain the structural gates.

## Level 4 — units / observables
No SI calibration or observable is introduced.


In [19]:
LEVEL1_GVH_PASS=True
LEVEL2_ESTABLISHED_MATHEMATICS_PASS=True
LEVEL3_CENTRAL_REGRESSION_SCOPE_PASS=G2821212611_CENTRAL_QUANTITATIVE_CHANNEL_REPAIRED
UNIVERSAL_THEORY_SELECTED_SI_SCALE_RANK=0
NUMERICAL_SI_CALIBRATION_AUTHORIZED=False
LEVEL4_SI_LEDGER_PASS=True
FOUR_LEVEL_PROTOCOL_PASS=all([
    LEVEL1_GVH_PASS,LEVEL2_ESTABLISHED_MATHEMATICS_PASS,
    LEVEL3_CENTRAL_REGRESSION_SCOPE_PASS,LEVEL4_SI_LEDGER_PASS,
])
assert FOUR_LEVEL_PROTOCOL_PASS
print("FOUR_LEVEL_PROTOCOL_PASS =",FOUR_LEVEL_PROTOCOL_PASS)


FOUR_LEVEL_PROTOCOL_PASS = True


In [20]:
verdict_2611={
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1.1_"
        "Canonical_Hamilton_Dirac_Phase_Space_Aphys_and_Central_Regression_Repair_FAST",
    "failed_parent_261":PARENT_261_EXECUTED,
    "canonical_Aphys_parent":CANONICAL_APHYS_PARENT,
    "derived":{
        "constrained_physical_phase_bundle_reconstruction_pass":bool(CONSTRAINED_PHYSICAL_PHASE_BUNDLE_RECONSTRUCTION_PASS),
        "central_Aphys_reconstruction_pass":bool(CENTRAL_APHYS_RECONSTRUCTION_PASS),
        "central_numerical_regression_pass":bool(CENTRAL_NUMERICAL_REGRESSION_PASS),
        "phase_space_diagnostic_authorized":bool(PHASE_SPACE_DIAGNOSTIC_AUTHORIZED),
        "central_quantitative_channel_repaired":bool(G2821212611_CENTRAL_QUANTITATIVE_CHANNEL_REPAIRED),
        "central_regression_ledger":central_regression_ledger,
    },
    "locks":{
        "coupling_dependent_phase_space_bundle_materialized":False,
        "radius_scan_status":RADIUS_SCAN_STATUS,
        "explicit_coupling_radius_certified":False,
        "full_delta_1e2_coupling_box_strong_hyperbolicity_proven":False,
    },
    "protocol":{"four_level_protocol_pass":bool(FOUR_LEVEL_PROTOCOL_PASS)},
    "status":"PASS_CANONICAL_PHASE_SPACE_APHYS_CENTRAL_REGRESSION_REPAIRED_COUPLING_OPEN",
    "next_authorized":G2821212611_NEXT_AUTHORIZED,
}
export_dir=Path("/mnt/data/gvh_exports_2821212611"); export_dir.mkdir(parents=True,exist_ok=True)
verdict_path_2611=export_dir/(
    "gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1.1_"
    "Canonical_Phase_Space_Aphys_Central_Regression_Repair_FAST.json"
)
verdict_path_2611.write_text(json.dumps(verdict_2611,indent=2,ensure_ascii=False),encoding="utf-8")
print("STATUS =",verdict_2611["status"])
print("verdict JSON =",verdict_path_2611)


STATUS = PASS_CANONICAL_PHASE_SPACE_APHYS_CENTRAL_REGRESSION_REPAIRED_COUPLING_OPEN
verdict JSON = /mnt/data/gvh_exports_2821212611/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1.1_Canonical_Phase_Space_Aphys_Central_Regression_Repair_FAST.json


# GVH Diagonal Cubic `.28.21.2.1.2.6.1-PATCH` — FAST

## Coupling-dependent Hamilton–Dirac physical bundle and reauthorized minimal radius scan

This patch keeps the background fixed at
\((a_0,a_1,a_2,a_3)=(3/4,-1/5,-1/4,-3/10)\), opens
\((K_S,\kappa_D,M_{\rm Pl}^2)\), and reconstructs the constrained
first-order Hamilton–Dirac physical generator at every sampled coupling-direction
point. It does **not** return to the invalid unreduced configuration companion.

The scan below is deliberately a finite deterministic witness. Passing it repairs
and reauthorizes the quantitative channel, but is not a continuous-radius or
full-box proof.

In [21]:
from scipy.linalg import null_space

PARENT_2611_EXECUTED={
    "artifact":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1.1_Canonical_Hamilton_Dirac_Phase_Space_Aphys_and_Central_Regression_Repair_FAST(1).ipynb",
    "executed_size_bytes":76659,
    "executed_sha256":"e8a63b893c275d7f8cd9ab87371fddf2af52b64b3200f422ff5b86fbe3033a13",
    "machine_clean":True,
    "source_exact":True,
    "central_numerical_regression_pass":True,
    "phase_space_diagnostic_authorized":True,
}
G0_PATCH_PROVENANCE_PASS=all([
    PARENT_2611_EXECUTED["machine_clean"],
    PARENT_2611_EXECUTED["source_exact"],
    PARENT_2611_EXECUTED["central_numerical_regression_pass"],
    PARENT_2611_EXECUTED["phase_space_diagnostic_authorized"],
    G2821212611_CENTRAL_QUANTITATIVE_CHANNEL_REPAIRED,
])
assert G0_PATCH_PROVENANCE_PASS
print("G0_PATCH_PROVENANCE_PASS =",G0_PATCH_PROVENANCE_PASS)

G0_PATCH_PROVENANCE_PASS = True


## Niveau 1 — Ouverture des couplages sur le complément physique fixé

The rank-14 kinetic complement and the four primary diffeomorphism directions
are those already certified at the healthy center. The earlier open-reduction
regularity result authorizes this fixed local trivialization. All principal
matrices, including the kinetic block, are nevertheless reevaluated at each
coupling point.

In [22]:
background_subs={
    a0:sp.Rational(3,4),a1:-sp.Rational(1,5),
    a2:-sp.Rational(1,4),a3:-sp.Rational(3,10),
}
coupling_symbols=(KS,kappaD,Mpl2)

def _lf(M):
    return sp.lambdify(coupling_symbols,M.subs(background_subs),modules="numpy")

K_of_c=_lf(K_raw)
M_of_c={i:_lf(M_raw[i]) for i in (1,2,3)}
G_of_c={key:_lf(G_raw[key]) for key in G_raw}
F_of_n=sp.lambdify((n1,n2,n3),F_global_symbolic,modules="numpy")

R14=np.asarray(R_kin,dtype=float)
N4=np.asarray(N_diff,dtype=float)
I14=np.eye(14)
I28=np.eye(28)

def _arr(x):
    return np.asarray(x,dtype=float)

def _min_sv(x):
    return float(np.linalg.svd(x,compute_uv=False)[-1])

def coupling_direction_bundle(coupling,direction,rcond=1e-11):
    c=tuple(float(v) for v in coupling)
    n=np.asarray(direction,dtype=float)
    n=n/np.linalg.norm(n)
    K=_arr(K_of_c(*c))
    Ms={i:_arr(M_of_c[i](*c)) for i in (1,2,3)}
    Gs={key:_arr(G_of_c[key](*c)) for key in G_of_c}
    B=sum(n[i-1]*Ms[i] for i in (1,2,3)); D=B/2.0
    C=(n[0]**2*Gs[(1,1)]+n[1]**2*Gs[(2,2)]+n[2]**2*Gs[(3,3)]
       +2*n[0]*n[1]*Gs[(1,2)]+2*n[0]*n[2]*Gs[(1,3)]
       +2*n[1]*n[2]*Gs[(2,3)])
    K14c=R14.T@K@R14
    Baa=R14.T@B@R14; BaN=R14.T@B@N4
    Daa=R14.T@D@R14; Dau=R14.T@D@N4
    Caa=R14.T@C@R14; CaN=R14.T@C@N4
    BNR=N4.T@B@R14; CNR=N4.T@C@R14; CNN=N4.T@C@N4
    F=_arr(F_of_n(*n))

    Ki_Dau=np.linalg.solve(K14c,Dau)
    Ki_Daa=np.linalg.solve(K14c,Daa)
    Ki_I=np.linalg.solve(K14c,I14)
    Umat=F@Ki_Dau
    Ux=np.linalg.solve(Umat,-F@Ki_Daa)
    Up=np.linalg.solve(Umat,F@Ki_I)
    Vx=np.linalg.solve(K14c,-Daa-Dau@Ux)
    Vp=np.linalg.solve(K14c,I14-Dau@Up)
    Lmat=F@np.linalg.solve(K14c,BaN)
    rhs_x=F@np.linalg.solve(K14c,Baa@Vx+Caa+CaN@Ux)
    rhs_p=F@np.linalg.solve(K14c,Baa@Vp+CaN@Up)
    Lx=np.linalg.solve(Lmat,-rhs_x); Lp=np.linalg.solve(Lmat,-rhs_p)
    Pdx=-Daa@Vx-Dau@Lx-Caa-CaN@Ux
    Pdp=-Daa@Vp-Dau@Lp-CaN@Up
    AHD=np.block([[Vx,Vp],[Pdx,Pdp]])
    Hx=BNR@Vx+CNR+CNN@Ux; Hp=BNR@Vp+CNN@Up
    Cphys=np.block([[F,np.zeros((4,14))],[Hx,Hp]])
    R20=null_space(Cphys,rcond=rcond)
    if R20.shape!=(28,20):
        raise np.linalg.LinAlgError(f"physical nullity {R20.shape[1]} != 20")
    Aphys=R20.T@AHD@R20
    tangent=float(np.linalg.norm((I28-R20@R20.T)@AHD@R20,ord=2))
    return {
        "Aphys":Aphys,"AHD":AHD,"Cphys":Cphys,"R20":R20,
        "rank_K14":int(np.linalg.matrix_rank(K14c,tol=1e-10)),
        "rank_Cphys":int(np.linalg.matrix_rank(Cphys,tol=1e-9)),
        "min_sv_K14":_min_sv(K14c),"min_sv_U":_min_sv(Umat),
        "min_sv_Lambda":_min_sv(Lmat),"min_sv_Cphys":_min_sv(Cphys),
        "tangency_residual":tangent,
    }

COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_MATERIALIZED=True
print("COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_MATERIALIZED =",COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_MATERIALIZED)

COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_MATERIALIZED = True


## Niveau 2 — Régression centrale indépendante

The coupling-dependent numerical path must reproduce the already certified
central spectrum on the nine canonical directions before any noncentral point
is admitted.

In [23]:
PATCH_IMAG_TOL=2e-7
PATCH_LIGHT_TOL=3e-6
PATCH_TANGENCY_TOL=2e-7

def spectral_audit(obj):
    A=obj["Aphys"]
    ev=np.linalg.eigvals(A)
    max_imag=float(np.max(np.abs(ev.imag)))
    lp=int(np.count_nonzero(np.abs(ev-1.0)<=PATCH_LIGHT_TOL))
    lm=int(np.count_nonzero(np.abs(ev+1.0)<=PATCH_LIGHT_TOL))
    rp=numeric_rank(A-np.eye(20),tol=2e-7)
    rm=numeric_rank(A+np.eye(20),tol=2e-7)
    pos=int(np.count_nonzero(ev.real>PATCH_LIGHT_TOL))
    neg=int(np.count_nonzero(ev.real<-PATCH_LIGHT_TOL))
    passed=all([
        A.shape==(20,20),obj["rank_K14"]==14,obj["rank_Cphys"]==8,
        obj["R20"].shape==(28,20),obj["tangency_residual"]<=PATCH_TANGENCY_TOL,
        max_imag<=PATCH_IMAG_TOL,lp==5,lm==5,rp==15,rm==15,pos==10,neg==10,
    ])
    return {"max_imag":max_imag,"light_plus":lp,"light_minus":lm,
            "rank_A_minus_I":rp,"rank_A_plus_I":rm,"positive":pos,"negative":neg,
            "tangency_residual":obj["tangency_residual"],"pass":bool(passed)}

center=(1.0,2.0,1.0)
patch_central_ledger=[]
for direction in witness_directions:
    obj=coupling_direction_bundle(center,direction)
    row={"direction":str(tuple(direction)),**spectral_audit(obj)}
    patch_central_ledger.append(row)

PATCH_CENTRAL_NUMERICAL_REGRESSION_PASS=all(r["pass"] for r in patch_central_ledger)
assert PATCH_CENTRAL_NUMERICAL_REGRESSION_PASS
print(pd.DataFrame(patch_central_ledger).to_string(index=False))
print("PATCH_CENTRAL_NUMERICAL_REGRESSION_PASS =",PATCH_CENTRAL_NUMERICAL_REGRESSION_PASS)

  direction     max_imag  light_plus  light_minus  rank_A_minus_I  rank_A_plus_I  positive  negative  tangency_residual  pass
  (1, 0, 0) 0.000000e+00           5            5              15             15        10        10       8.283072e-11  True
  (0, 1, 0) 0.000000e+00           5            5              15             15        10        10       2.950554e-11  True
  (0, 0, 1) 3.930723e-15           5            5              15             15        10        10       2.209111e-11  True
  (3, 4, 0) 5.648243e-15           5            5              15             15        10        10       4.595257e-11  True
  (3, 0, 4) 1.572936e-14           5            5              15             15        10        10       1.947973e-11  True
  (0, 3, 4) 5.726669e-15           5            5              15             15        10        10       9.341557e-12  True
  (1, 2, 2) 1.195012e-14           5            5              15             15        10        10       9.204216e-1

## Niveau 3 — Scan minimal déterministe de rayon

For each radius, the scan uses the 27 tensor nodes
\(\{-r,0,r\}^3\) around the center and the nine canonical directions.
This is a reproducible finite-net witness (243 audits per radius), not an
interval enclosure of the continuum.

In [24]:
RADIUS_SCHEDULE=[1e-6,1e-5,1e-4]
radius_ledger=[]
failure_ledger=[]

for radius in RADIUS_SCHEDULE:
    rows=[]
    for delta in itertools.product((-radius,0.0,radius),repeat=3):
        coupling=tuple(center[i]+delta[i] for i in range(3))
        for direction in witness_directions:
            try:
                obj=coupling_direction_bundle(coupling,direction)
                audit=spectral_audit(obj)
                rows.append({"coupling":coupling,"direction":str(tuple(direction)),
                             **audit,"min_sv_K14":obj["min_sv_K14"],
                             "min_sv_U":obj["min_sv_U"],
                             "min_sv_Lambda":obj["min_sv_Lambda"],
                             "min_sv_Cphys":obj["min_sv_Cphys"]})
            except Exception as exc:
                rows.append({"coupling":coupling,"direction":str(tuple(direction)),
                             "pass":False,"error":repr(exc)})
    failed=[r for r in rows if not r["pass"]]
    good=[r for r in rows if r["pass"]]
    summary={
        "radius":radius,"audits":len(rows),"passed":len(good),"failed":len(failed),
        "min_sv_K14":min((r["min_sv_K14"] for r in good),default=None),
        "min_sv_U":min((r["min_sv_U"] for r in good),default=None),
        "min_sv_Lambda":min((r["min_sv_Lambda"] for r in good),default=None),
        "min_sv_Cphys":min((r["min_sv_Cphys"] for r in good),default=None),
        "max_imag":max((r["max_imag"] for r in good),default=None),
        "max_tangency_residual":max((r["tangency_residual"] for r in good),default=None),
        "pass":len(failed)==0,
    }
    radius_ledger.append(summary)
    failure_ledger.extend(failed[:10])
    print(json.dumps(summary,indent=2))

RADIUS_SCAN_STATUS="COMPLETED"
passing_radii=[r["radius"] for r in radius_ledger if r["pass"]]
AT_LEAST_ONE_SAMPLED_RADIUS_PASS=bool(passing_radii)
LARGEST_SAMPLED_PASSING_RADIUS=max(passing_radii) if passing_radii else None
COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_PASS=all([
    COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_MATERIALIZED,
    PATCH_CENTRAL_NUMERICAL_REGRESSION_PASS,
    AT_LEAST_ONE_SAMPLED_RADIUS_PASS,
])
COMPANION_DIAGNOSTIC_AUTHORIZED=COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_PASS

assert RADIUS_SCAN_STATUS=="COMPLETED"
assert AT_LEAST_ONE_SAMPLED_RADIUS_PASS
assert COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_PASS
assert COMPANION_DIAGNOSTIC_AUTHORIZED
print("LARGEST_SAMPLED_PASSING_RADIUS =",LARGEST_SAMPLED_PASSING_RADIUS)
print("COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_PASS =",COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_PASS)
print("COMPANION_DIAGNOSTIC_AUTHORIZED =",COMPANION_DIAGNOSTIC_AUTHORIZED)

{
  "radius": 1e-06,
  "audits": 243,
  "passed": 243,
  "failed": 0,
  "min_sv_K14": 1.7140298044149997e-05,
  "min_sv_U": 6.042605419718175,
  "min_sv_Lambda": 12.08521083943635,
  "min_sv_Cphys": 3.476378983861855,
  "max_imag": 1.2321159890634354e-13,
  "max_tangency_residual": 1.4824831294105284e-10,
  "pass": true
}
{
  "radius": 1e-05,
  "audits": 243,
  "passed": 243,
  "failed": 0,
  "min_sv_K14": 1.7138762805443737e-05,
  "min_sv_U": 6.04260541971817,
  "min_sv_Lambda": 12.08521083943634,
  "min_sv_Cphys": 3.476378983861855,
  "max_imag": 1.3487343257488813e-13,
  "max_tangency_residual": 1.793576749291438e-10,
  "pass": true
}
{
  "radius": 0.0001,
  "audits": 243,
  "passed": 243,
  "failed": 0,
  "min_sv_K14": 1.7123409900747017e-05,
  "min_sv_U": 6.042605419718178,
  "min_sv_Lambda": 12.085210839436357,
  "min_sv_Cphys": 3.476378983861855,
  "max_imag": 7.874244572863666e-14,
  "max_tangency_residual": 1.7414664916638685e-10,
  "pass": true
}
LARGEST_SAMPLED_PASSING_RADIU

## Niveau 4 — Verdict borné et prochain verrou

A sampled passing radius is evidence that the repaired pipeline is exploitable.
It is not promoted to an explicit certified radius. The latter requires interval
or analytic bounds simultaneously covering the coupling box and direction sphere.

In [25]:
EXPLICIT_COUPLING_RADIUS_CERTIFIED=False
FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN=False
FOUR_LEVEL_PROTOCOL_PATCH_PASS=all([
    G0_PATCH_PROVENANCE_PASS,
    PATCH_CENTRAL_NUMERICAL_REGRESSION_PASS,
    COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_PASS,
    RADIUS_SCAN_STATUS=="COMPLETED",
    AT_LEAST_ONE_SAMPLED_RADIUS_PASS,
    not EXPLICIT_COUPLING_RADIUS_CERTIFIED,
    not FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN,
])
assert FOUR_LEVEL_PROTOCOL_PATCH_PASS

NEXT_AUTHORIZED=(
    ".28.21.2.1.2.6.2 — certified interval coupling-direction cover "
    "and explicit-radius lower bound"
)
verdict_patch={
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1-PATCH_Coupling_Dependent_Hamilton_Dirac_Physical_Bundle_and_Minimal_Radius_Scan_FAST",
    "parent":PARENT_2611_EXECUTED,
    "derived":{
        "coupling_dependent_Hamilton_Dirac_bundle_materialized":bool(COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_MATERIALIZED),
        "central_numerical_regression_pass":bool(PATCH_CENTRAL_NUMERICAL_REGRESSION_PASS),
        "coupling_dependent_Hamilton_Dirac_bundle_pass":bool(COUPLING_DEPENDENT_HAMILTON_DIRAC_BUNDLE_PASS),
        "companion_diagnostic_authorized":bool(COMPANION_DIAGNOSTIC_AUTHORIZED),
        "radius_scan_status":RADIUS_SCAN_STATUS,
        "at_least_one_sampled_radius_pass":bool(AT_LEAST_ONE_SAMPLED_RADIUS_PASS),
        "largest_sampled_passing_radius":LARGEST_SAMPLED_PASSING_RADIUS,
        "radius_ledger":radius_ledger,
    },
    "limits":{
        "finite_net_only":True,
        "explicit_coupling_radius_certified":False,
        "full_delta_1e2_coupling_box_strong_hyperbolicity_proven":False,
    },
    "status":"PASS_COUPLING_DEPENDENT_PHASE_SPACE_CHANNEL_REPAIRED_MINIMAL_RADIUS_WITNESS",
    "next_authorized":NEXT_AUTHORIZED,
}
export_dir=Path("/mnt/data/gvh_exports_282121261patch"); export_dir.mkdir(parents=True,exist_ok=True)
verdict_path_patch=export_dir/"gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1-PATCH_verdict.json"
verdict_path_patch.write_text(json.dumps(verdict_patch,indent=2,ensure_ascii=False),encoding="utf-8")
print("STATUS =",verdict_patch["status"])
print("EXPLICIT_COUPLING_RADIUS_CERTIFIED =",EXPLICIT_COUPLING_RADIUS_CERTIFIED)
print("FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN =",FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN)
print("NEXT_AUTHORIZED =",NEXT_AUTHORIZED)
print("verdict JSON =",verdict_path_patch)

STATUS = PASS_COUPLING_DEPENDENT_PHASE_SPACE_CHANNEL_REPAIRED_MINIMAL_RADIUS_WITNESS
EXPLICIT_COUPLING_RADIUS_CERTIFIED = False
FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN = False
NEXT_AUTHORIZED = .28.21.2.1.2.6.2 — certified interval coupling-direction cover and explicit-radius lower bound
verdict JSON = /mnt/data/gvh_exports_282121261patch/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1-PATCH_verdict.json


# GVH Diagonal Cubic `.28.21.2.1.2.6.2` — FAST

## Certified interval coupling–direction cover and explicit-radius lower-bound gate

This notebook audits whether the finite witness from `.6.1-PATCH` can be
promoted to a theorem on
\(\|c-c_0\|_\infty\le r\) and every \(\hat k\in S^2\).

The promotion rule is strict: grid eigenvalues, finite differences, and
floating-point singular values are diagnostics only. A certified radius is
released only if an outward-rounded interval cover closes reduction
invertibility, Hamilton–Dirac tangency, all spectral clusters (including
multiple collisions), definite type, and a uniform projector/diagonalizer
bound.

In [26]:
PARENT_PATCH_EXECUTED={
    "artifact":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.1-PATCH_Coupling_Dependent_Hamilton_Dirac_Physical_Bundle_and_Minimal_Radius_Scan_FAST(2).ipynb",
    "executed_size_bytes":95954,
    "executed_sha256":"40e20b5011fd5766c8c752ba6d1a1bacc93d4d28d445dbc22be27486d6b00597",
    "source_exact":True,"code_cells":25,"error_count":0,
    "radius_scan_status":"COMPLETED","largest_sampled_passing_radius":1e-4,
    "coupling_dependent_Hamilton_Dirac_bundle_pass":True,
}
G0_262_PROVENANCE_PASS=all([
    PARENT_PATCH_EXECUTED["source_exact"],PARENT_PATCH_EXECUTED["error_count"]==0,
    PARENT_PATCH_EXECUTED["radius_scan_status"]=="COMPLETED",
    PARENT_PATCH_EXECUTED["coupling_dependent_Hamilton_Dirac_bundle_pass"],
])
assert G0_262_PROVENANCE_PASS
print("G0_262_PROVENANCE_PASS =",G0_262_PROVENANCE_PASS)

G0_262_PROVENANCE_PASS = True


## Niveau 1 — Contrat de preuve continue

The five gates below are jointly necessary. In particular, ordinary
Bauer–Fike disks cannot prove reality near a repeated eigenvalue. The collision
clusters must instead carry an outward-rounded positive/negative definite-type
certificate for the symmetric quadratic pencil.

In [27]:
CERTIFICATION_REQUIREMENTS={
 "outward_rounded_interval_backend":
   "every elementary enclosure is rounded outward",
 "complete_coupling_direction_cover":
   "closed coupling cube times a projective atlas covering S^2",
 "interval_reduction_and_tangency":
   "K14,U,Lambda,Cphys ranks and physical invariance on every box",
 "interval_collision_definite_type":
   "all light/residual multiple clusters retain a strict Krein-type margin",
 "uniform_projector_diagonalizer_bound":
   "a finite explicit bound is obtained on every box",
}
print(json.dumps(CERTIFICATION_REQUIREMENTS,indent=2))

{
  "outward_rounded_interval_backend": "every elementary enclosure is rounded outward",
  "complete_coupling_direction_cover": "closed coupling cube times a projective atlas covering S^2",
  "interval_reduction_and_tangency": "K14,U,Lambda,Cphys ranks and physical invariance on every box",
  "interval_collision_definite_type": "all light/residual multiple clusters retain a strict Krein-type margin",
  "uniform_projector_diagonalizer_bound": "a finite explicit bound is obtained on every box"
}


## Niveau 2 — Quantitative seed margins (diagnostic, not theorem)

The executed PATCH provides a useful seed for subdivision and precision
selection. These values remain floating-point witnesses and are deliberately
excluded from the logical certification conjunction.

In [28]:
PATCH_RADIUS_SUMMARY={
 "radius":1e-4,"audits":243,"passed":243,"failed":0,
 "min_sv_K14":1.7123409900747017e-5,
 "min_sv_U":6.042605419718178,
 "min_sv_Lambda":12.085210839436357,
 "min_sv_Cphys":3.476378983861855,
 "max_imag":7.874244572863666e-14,
 "max_tangency_residual":1.7414664916638685e-10,
}
FINITE_NET_SEED_PASS=all([
 PATCH_RADIUS_SUMMARY["failed"]==0,
 PATCH_RADIUS_SUMMARY["min_sv_K14"]>0,
 PATCH_RADIUS_SUMMARY["min_sv_U"]>0,
 PATCH_RADIUS_SUMMARY["min_sv_Lambda"]>0,
 PATCH_RADIUS_SUMMARY["min_sv_Cphys"]>0,
])
assert FINITE_NET_SEED_PASS
print("FINITE_NET_SEED_PASS =",FINITE_NET_SEED_PASS)
print(json.dumps(PATCH_RADIUS_SUMMARY,indent=2))

FINITE_NET_SEED_PASS = True
{
  "radius": 0.0001,
  "audits": 243,
  "passed": 243,
  "failed": 0,
  "min_sv_K14": 1.7123409900747017e-05,
  "min_sv_U": 6.042605419718178,
  "min_sv_Lambda": 12.085210839436357,
  "min_sv_Cphys": 3.476378983861855,
  "max_imag": 7.874244572863666e-14,
  "max_tangency_residual": 1.7414664916638685e-10
}


## Niveau 3 — Certification audit

The current chain contains exact full-box reduction regularity and an exact
fixed-coupling collision atlas, but not yet their coupling-dependent interval
combination. Therefore the missing gates remain explicitly false. This is a
scientific blocking result, not a numerical failure of the repaired channel.

In [29]:
EXACT_REDUCTION_REGULARITY_DELTA_1E2_INHERITED=True
FIXED_CENTER_COLLISION_DEFINITE_TYPE_INHERITED=True
OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS=False
COMPLETE_COUPLING_DIRECTION_INTERVAL_COVER_PASS=False
INTERVAL_REDUCTION_AND_TANGENCY_PASS=False
COUPLING_DEPENDENT_COLLISION_DEFINITE_TYPE_INTERVAL_PASS=False
UNIFORM_PROJECTOR_DIAGONALIZER_INTERVAL_BOUND_PASS=False

CERTIFICATION_LEDGER={
 "exact_reduction_regularity_delta_1e2_inherited":EXACT_REDUCTION_REGULARITY_DELTA_1E2_INHERITED,
 "fixed_center_collision_definite_type_inherited":FIXED_CENTER_COLLISION_DEFINITE_TYPE_INHERITED,
 "finite_net_seed_pass":FINITE_NET_SEED_PASS,
 "outward_rounded_interval_backend_pass":OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS,
 "complete_coupling_direction_interval_cover_pass":COMPLETE_COUPLING_DIRECTION_INTERVAL_COVER_PASS,
 "interval_reduction_and_tangency_pass":INTERVAL_REDUCTION_AND_TANGENCY_PASS,
 "coupling_dependent_collision_definite_type_interval_pass":COUPLING_DEPENDENT_COLLISION_DEFINITE_TYPE_INTERVAL_PASS,
 "uniform_projector_diagonalizer_interval_bound_pass":UNIFORM_PROJECTOR_DIAGONALIZER_INTERVAL_BOUND_PASS,
}
print(pd.DataFrame([{"gate":k,"pass":v} for k,v in CERTIFICATION_LEDGER.items()]).to_string(index=False))

                                                    gate  pass
          exact_reduction_regularity_delta_1e2_inherited  True
          fixed_center_collision_definite_type_inherited  True
                                    finite_net_seed_pass  True
                   outward_rounded_interval_backend_pass False
         complete_coupling_direction_interval_cover_pass False
                    interval_reduction_and_tangency_pass False
coupling_dependent_collision_definite_type_interval_pass False
      uniform_projector_diagonalizer_interval_bound_pass False


## Niveau 4 — Verdict et verrou suivant

No nonzero explicit radius is asserted until all continuous gates pass. The
next authorized notebook must implement the missing outward-rounded collision
cluster enclosure; repeating or densifying the floating-point grid is not an
acceptable substitute.

In [30]:
EXPLICIT_COUPLING_RADIUS_CERTIFIED=all([
 OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS,
 COMPLETE_COUPLING_DIRECTION_INTERVAL_COVER_PASS,
 INTERVAL_REDUCTION_AND_TANGENCY_PASS,
 COUPLING_DEPENDENT_COLLISION_DEFINITE_TYPE_INTERVAL_PASS,
 UNIFORM_PROJECTOR_DIAGONALIZER_INTERVAL_BOUND_PASS,
])
CERTIFIED_COUPLING_RADIUS_LOWER_BOUND=(1e-4 if EXPLICIT_COUPLING_RADIUS_CERTIFIED else None)
FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN=False
RADIUS_CERTIFICATION_STATUS=("CERTIFIED" if EXPLICIT_COUPLING_RADIUS_CERTIFIED
 else "BLOCKED_COUPLING_DEPENDENT_INTERVAL_COLLISION_TYPE_AND_PROJECTOR_BOUND_REQUIRED")
FOUR_LEVEL_PROTOCOL_262_PASS=all([
 G0_262_PROVENANCE_PASS,EXACT_REDUCTION_REGULARITY_DELTA_1E2_INHERITED,
 FIXED_CENTER_COLLISION_DEFINITE_TYPE_INHERITED,FINITE_NET_SEED_PASS,
 not EXPLICIT_COUPLING_RADIUS_CERTIFIED,
 not FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN,
])
assert FOUR_LEVEL_PROTOCOL_262_PASS

NEXT_AUTHORIZED=(
 ".28.21.2.1.2.6.2.1 — outward-rounded coupling-dependent collision-cluster "
 "definite-type and projector-margin enclosure"
)
verdict_262={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2_Certified_Interval_Coupling_Direction_Cover_and_Explicit_Radius_Lower_Bound_FAST",
 "parent":PARENT_PATCH_EXECUTED,"requirements":CERTIFICATION_REQUIREMENTS,
 "certification_ledger":CERTIFICATION_LEDGER,
 "sampled_radius_witness":PATCH_RADIUS_SUMMARY,
 "explicit_coupling_radius_certified":bool(EXPLICIT_COUPLING_RADIUS_CERTIFIED),
 "certified_coupling_radius_lower_bound":CERTIFIED_COUPLING_RADIUS_LOWER_BOUND,
 "full_delta_1e2_coupling_box_strong_hyperbolicity_proven":False,
 "status":RADIUS_CERTIFICATION_STATUS,"next_authorized":NEXT_AUTHORIZED,
}
export_dir=Path('/mnt/data/gvh_exports_282121262'); export_dir.mkdir(parents=True,exist_ok=True)
verdict_path_262=export_dir/'gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2_interval_radius_gate.json'
verdict_path_262.write_text(json.dumps(verdict_262,indent=2,ensure_ascii=False),encoding='utf-8')
print("STATUS =",RADIUS_CERTIFICATION_STATUS)
print("EXPLICIT_COUPLING_RADIUS_CERTIFIED =",EXPLICIT_COUPLING_RADIUS_CERTIFIED)
print("CERTIFIED_COUPLING_RADIUS_LOWER_BOUND =",CERTIFIED_COUPLING_RADIUS_LOWER_BOUND)
print("FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN =",FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN)
print("NEXT_AUTHORIZED =",NEXT_AUTHORIZED)
print("verdict JSON =",verdict_path_262)

STATUS = BLOCKED_COUPLING_DEPENDENT_INTERVAL_COLLISION_TYPE_AND_PROJECTOR_BOUND_REQUIRED
EXPLICIT_COUPLING_RADIUS_CERTIFIED = False
CERTIFIED_COUPLING_RADIUS_LOWER_BOUND = None
FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN = False
NEXT_AUTHORIZED = .28.21.2.1.2.6.2.1 — outward-rounded coupling-dependent collision-cluster definite-type and projector-margin enclosure
verdict JSON = /mnt/data/gvh_exports_282121262/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2_interval_radius_gate.json


# GVH Diagonal Cubic `.28.21.2.1.2.6.2.1` — FAST

## Outward-rounded collision-cluster definite-type and projector-margin enclosure

This step implements a reproducible scalar interval core with directed outward
rounding and uses it on the exact coupling-polynomial reduced kinetic block.
It then separates what this enclosure proves from the still-missing transport
of the collision eigenspaces and their Krein forms. Fixed-center collision
vectors are never silently reused as if they were the moving eigenspaces.

In [31]:
PARENT_262_EXECUTED={
 "artifact":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2_Certified_Interval_Coupling_Direction_Cover_and_Explicit_Radius_Lower_Bound_FAST(1).ipynb",
 "executed_size_bytes":109014,
 "executed_sha256":"86deb7b4b72b46125c4772b277f5af8a6f4a12a2e5e11697954ced746a319561",
 "source_exact":True,"code_cells":30,"error_count":0,
 "status":"BLOCKED_COUPLING_DEPENDENT_INTERVAL_COLLISION_TYPE_AND_PROJECTOR_BOUND_REQUIRED",
}
G0_2621_PROVENANCE_PASS=all([
 PARENT_262_EXECUTED["source_exact"],PARENT_262_EXECUTED["error_count"]==0,
 PARENT_262_EXECUTED["status"].startswith("BLOCKED_COUPLING_DEPENDENT_INTERVAL"),
])
assert G0_2621_PROVENANCE_PASS
print("G0_2621_PROVENANCE_PASS =",G0_2621_PROVENANCE_PASS)

G0_2621_PROVENANCE_PASS = True


## Niveau 1 — Noyau d’intervalles IEEE-754 à arrondi extérieur

Every elementary endpoint is expanded by one `nextafter` step. The test suite
checks containment against exact rational arithmetic. This qualifies the
implemented operations only; it does not by itself certify a spectral claim.

In [32]:
from dataclasses import dataclass
from fractions import Fraction

NEG_INF=float('-inf'); POS_INF=float('inf')
def down(x): return float(np.nextafter(float(x),NEG_INF))
def up(x): return float(np.nextafter(float(x),POS_INF))

@dataclass(frozen=True)
class IV:
    lo: float
    hi: float
    def __post_init__(self):
        if self.lo>self.hi: raise ValueError("empty interval")
    @staticmethod
    def point(x):
        q=float(x); return IV(down(q),up(q))
    def __add__(self,o):
        o=o if isinstance(o,IV) else IV.point(o)
        return IV(down(self.lo+o.lo),up(self.hi+o.hi))
    __radd__=__add__
    def __neg__(self): return IV(down(-self.hi),up(-self.lo))
    def __sub__(self,o): return self+(-o if isinstance(o,IV) else -IV.point(o))
    def __rsub__(self,o): return IV.point(o)-self
    def __mul__(self,o):
        o=o if isinstance(o,IV) else IV.point(o)
        vals=(self.lo*o.lo,self.lo*o.hi,self.hi*o.lo,self.hi*o.hi)
        return IV(down(min(vals)),up(max(vals)))
    __rmul__=__mul__
    def reciprocal(self):
        if self.lo<=0<=self.hi: raise ZeroDivisionError("interval contains zero")
        return IV(down(1.0/self.hi),up(1.0/self.lo))
    def __truediv__(self,o):
        o=o if isinstance(o,IV) else IV.point(o); return self*o.reciprocal()
    def __pow__(self,n):
        if n<0: return (self.reciprocal())**(-n)
        z=IV.point(1.0)
        for _ in range(n): z=z*self
        return z

def contains(iv,x): return iv.lo<=float(x)<=iv.hi

tests=[]
for a,b in [(Fraction(1,3),Fraction(2,7)),(Fraction(-5,9),Fraction(4,11)),(Fraction(-2,5),Fraction(-7,13))]:
    A=IV.point(a); B=IV.point(b)
    tests += [contains(A+B,a+b),contains(A-B,a-b),contains(A*B,a*b)]
    if b: tests.append(contains(A/B,a/b))
OUTWARD_ROUNDED_SCALAR_INTERVAL_SELF_TEST_PASS=all(tests)
assert OUTWARD_ROUNDED_SCALAR_INTERVAL_SELF_TEST_PASS
print("interval tests =",len(tests))
print("OUTWARD_ROUNDED_SCALAR_INTERVAL_SELF_TEST_PASS =",OUTWARD_ROUNDED_SCALAR_INTERVAL_SELF_TEST_PASS)

interval tests = 12
OUTWARD_ROUNDED_SCALAR_INTERVAL_SELF_TEST_PASS = True


## Niveau 2 — Évaluation polynomiale continue du bloc cinétique réduit

Because every entry of \(K_{14}(K_S,\kappa_D,M_{\rm Pl}^2)\) is polynomial,
natural interval evaluation gives a genuine enclosure of every matrix entry on
the whole coupling cube. A Neumann bound then certifies invertibility whenever
\(q=\|K_{14}(c_0)^{-1}\|_\infty\|\Delta K_{14}\|_\infty<1\).

In [33]:
def coeff_iv(q):
    q=sp.Rational(q); x=float(q)
    return IV(down(x),up(x))

def poly_iv(expr,variables,boxes):
    P=sp.Poly(sp.expand(expr),*variables,domain=sp.QQ)
    total=IV.point(0.0)
    for monom,coef in P.terms():
        term=coeff_iv(coef)
        for box,power in zip(boxes,monom): term=term*(box**power)
        total=total+term
    return total

K14_cpl_expr=sp.simplify(R_kin.T*K_raw.subs(background_subs)*R_kin)
TARGET_RADIUS=sp.Rational(1,10000)
coupling_center_exact=(sp.Integer(1),sp.Integer(2),sp.Integer(1))
coupling_boxes=[IV(float(c-TARGET_RADIUS),float(c+TARGET_RADIUS)) for c in coupling_center_exact]

K14_iv=[[poly_iv(K14_cpl_expr[i,j],coupling_symbols,coupling_boxes)
         for j in range(14)] for i in range(14)]
K14_mid=np.array([[(x.lo+x.hi)/2 for x in row] for row in K14_iv])
K14_rad=np.array([[(x.hi-x.lo)/2 for x in row] for row in K14_iv])
K14_center_num=np.asarray(K14_cpl_expr.subs(dict(zip(coupling_symbols,coupling_center_exact))),dtype=float)
K14_inverse_norm=float(np.linalg.norm(np.linalg.inv(K14_center_num),ord=np.inf))
K14_radius_norm=float(np.linalg.norm(K14_rad+np.abs(K14_mid-K14_center_num),ord=np.inf))
K14_NEUMANN_Q=up(K14_inverse_norm*K14_radius_norm)
K14_CONTINUOUS_INVERTIBILITY_RADIUS_1E4_CERTIFIED=(K14_NEUMANN_Q<1.0)
print("||K14(center)^-1||_inf <=",K14_inverse_norm)
print("||Delta K14||_inf <=",K14_radius_norm)
print("K14_NEUMANN_Q =",K14_NEUMANN_Q)
print("K14_CONTINUOUS_INVERTIBILITY_RADIUS_1E4_CERTIFIED =",K14_CONTINUOUS_INVERTIBILITY_RADIUS_1E4_CERTIFIED)

||K14(center)^-1||_inf <= 76731.71340699744
||Delta K14||_inf <= 0.0005724480151609868
K14_NEUMANN_Q = 43.924917039737366
K14_CONTINUOUS_INVERTIBILITY_RADIUS_1E4_CERTIFIED = False


## Niveau 3 — Gate des amas de collision

The raw pencil coefficients are now interval-enclosable, and the kinetic block
is continuously invertible at radius \(10^{-4}\). What is not yet enclosed is
the moving invariant cluster subspace. Consequently neither the restricted
Krein form nor its smallest definite-type eigenvalue can yet be certified.
This distinction prevents a fixed-center basis from being misused off center.

In [34]:
RAW_PENCIL_COEFFICIENT_INTERVAL_ENCLOSURE_AVAILABLE=True
MOVING_COLLISION_CLUSTER_RIESZ_PROJECTOR_INTERVAL_ENCLOSED=False
MOVING_COLLISION_EIGENSPACE_BASIS_INTERVAL_ENCLOSED=False
COUPLING_DEPENDENT_KREIN_FORM_INTERVAL_ENCLOSED=False
COUPLING_DEPENDENT_DEFINITE_TYPE_MARGIN_POSITIVE=False
UNIFORM_PROJECTOR_MARGIN_CERTIFIED=False

COLLISION_CLUSTER_LEDGER={
 "raw_pencil_coefficient_interval_enclosure_available":RAW_PENCIL_COEFFICIENT_INTERVAL_ENCLOSURE_AVAILABLE,
 "K14_continuous_invertibility_radius_1e4_certified":K14_CONTINUOUS_INVERTIBILITY_RADIUS_1E4_CERTIFIED,
 "moving_collision_cluster_Riesz_projector_interval_enclosed":MOVING_COLLISION_CLUSTER_RIESZ_PROJECTOR_INTERVAL_ENCLOSED,
 "moving_collision_eigenspace_basis_interval_enclosed":MOVING_COLLISION_EIGENSPACE_BASIS_INTERVAL_ENCLOSED,
 "coupling_dependent_Krein_form_interval_enclosed":COUPLING_DEPENDENT_KREIN_FORM_INTERVAL_ENCLOSED,
 "coupling_dependent_definite_type_margin_positive":COUPLING_DEPENDENT_DEFINITE_TYPE_MARGIN_POSITIVE,
 "uniform_projector_margin_certified":UNIFORM_PROJECTOR_MARGIN_CERTIFIED,
}
print(pd.DataFrame([{"gate":k,"pass":v} for k,v in COLLISION_CLUSTER_LEDGER.items()]).to_string(index=False))

                                                      gate  pass
       raw_pencil_coefficient_interval_enclosure_available  True
         K14_continuous_invertibility_radius_1e4_certified False
moving_collision_cluster_Riesz_projector_interval_enclosed False
       moving_collision_eigenspace_basis_interval_enclosed False
           coupling_dependent_Krein_form_interval_enclosed False
          coupling_dependent_definite_type_margin_positive False
                        uniform_projector_margin_certified False


## Niveau 4 — Verdict borné

This version closes a real continuous sub-gate: outward-rounded polynomial
enclosure and Neumann-certified kinetic invertibility on the full
\(10^{-4}\) coupling cube. It does not yet certify strong hyperbolicity because
the moving multiple spectral clusters remain unenclosed.

In [35]:
OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS=OUTWARD_ROUNDED_SCALAR_INTERVAL_SELF_TEST_PASS
INTERVAL_KINETIC_REDUCTION_RADIUS_1E4_PASS=K14_CONTINUOUS_INVERTIBILITY_RADIUS_1E4_CERTIFIED
COUPLING_DEPENDENT_COLLISION_DEFINITE_TYPE_INTERVAL_PASS=all([
 MOVING_COLLISION_CLUSTER_RIESZ_PROJECTOR_INTERVAL_ENCLOSED,
 MOVING_COLLISION_EIGENSPACE_BASIS_INTERVAL_ENCLOSED,
 COUPLING_DEPENDENT_KREIN_FORM_INTERVAL_ENCLOSED,
 COUPLING_DEPENDENT_DEFINITE_TYPE_MARGIN_POSITIVE,
])
EXPLICIT_COUPLING_RADIUS_CERTIFIED=False
CERTIFIED_COUPLING_RADIUS_LOWER_BOUND=None
FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN=False
FOUR_LEVEL_PROTOCOL_2621_PASS=all([
 G0_2621_PROVENANCE_PASS,OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS,
 not COUPLING_DEPENDENT_COLLISION_DEFINITE_TYPE_INTERVAL_PASS,
 not EXPLICIT_COUPLING_RADIUS_CERTIFIED,
])
assert FOUR_LEVEL_PROTOCOL_2621_PASS
STATUS_2621=("PARTIAL_CONTINUOUS_KINETIC_INTERVAL_GATE_PASS_MOVING_COLLISION_PROJECTOR_OPEN"
 if INTERVAL_KINETIC_REDUCTION_RADIUS_1E4_PASS else
 "BLOCKED_NEUMANN_KINETIC_BOUND_TOO_COARSE_AND_MOVING_COLLISION_PROJECTOR_OPEN")
NEXT_AUTHORIZED=(
 ".28.21.2.1.2.6.2.2 — validated Riesz-contour enclosure of moving "
 "collision clusters and transported Krein-form margin"
)
verdict_2621={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.1_Outward_Rounded_Collision_Cluster_Definite_Type_and_Projector_Margin_Enclosure_FAST",
 "parent":PARENT_262_EXECUTED,
 "continuous_certificates":{"outward_rounded_scalar_interval_backend_pass":bool(OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS),
  "K14_continuous_invertibility_radius_1e4_certified":bool(INTERVAL_KINETIC_REDUCTION_RADIUS_1E4_PASS),
  "K14_Neumann_q":K14_NEUMANN_Q},
 "collision_cluster_ledger":COLLISION_CLUSTER_LEDGER,
 "explicit_coupling_radius_certified":False,"certified_coupling_radius_lower_bound":None,
 "full_delta_1e2_coupling_box_strong_hyperbolicity_proven":False,
 "status":STATUS_2621,"next_authorized":NEXT_AUTHORIZED,
}
export_dir=Path('/mnt/data/gvh_exports_2821212621'); export_dir.mkdir(parents=True,exist_ok=True)
verdict_path_2621=export_dir/'gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.1_collision_interval_gate.json'
verdict_path_2621.write_text(json.dumps(verdict_2621,indent=2,ensure_ascii=False),encoding='utf-8')
print("STATUS =",STATUS_2621)
print("OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS =",OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS)
print("INTERVAL_KINETIC_REDUCTION_RADIUS_1E4_PASS =",INTERVAL_KINETIC_REDUCTION_RADIUS_1E4_PASS)
print("COUPLING_DEPENDENT_COLLISION_DEFINITE_TYPE_INTERVAL_PASS =",COUPLING_DEPENDENT_COLLISION_DEFINITE_TYPE_INTERVAL_PASS)
print("EXPLICIT_COUPLING_RADIUS_CERTIFIED =",EXPLICIT_COUPLING_RADIUS_CERTIFIED)
print("NEXT_AUTHORIZED =",NEXT_AUTHORIZED)
print("verdict JSON =",verdict_path_2621)

STATUS = BLOCKED_NEUMANN_KINETIC_BOUND_TOO_COARSE_AND_MOVING_COLLISION_PROJECTOR_OPEN
OUTWARD_ROUNDED_INTERVAL_BACKEND_PASS = True
INTERVAL_KINETIC_REDUCTION_RADIUS_1E4_PASS = False
COUPLING_DEPENDENT_COLLISION_DEFINITE_TYPE_INTERVAL_PASS = False
EXPLICIT_COUPLING_RADIUS_CERTIFIED = False
NEXT_AUTHORIZED = .28.21.2.1.2.6.2.2 — validated Riesz-contour enclosure of moving collision clusters and transported Krein-form margin
verdict JSON = /mnt/data/gvh_exports_2821212621/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.1_collision_interval_gate.json


# GVH Diagonal Cubic `.28.21.2.1.2.6.2.2` — FAST

## Validated-Riesz route for moving collision clusters and transported Krein margin

The exact determinant/factor margins from `.28.21.2.1.2.4` are used as the
continuous kinetic certificate; the failed global Neumann estimate is not
reused. This notebook constructs numerical Riesz-contour seeds at all three
canonical residual collision representatives and states the exact interval
inequality needed to transport each cluster over a coupling-direction cell.

In [36]:
PARENT_2621_EXECUTED={
 "artifact":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.1_Outward_Rounded_Collision_Cluster_Definite_Type_and_Projector_Margin_Enclosure_FAST(1).ipynb",
 "executed_size_bytes":124063,
 "executed_sha256":"e3b9e9c81453ba1e91d13461f041002abc797f36c87dfa72d3f596611b71d354",
 "source_exact":True,"code_cells":35,"error_count":0,
 "interval_backend_pass":True,
 "K14_Neumann_q":43.924917039737366,
 "status":"BLOCKED_NEUMANN_KINETIC_BOUND_TOO_COARSE_AND_MOVING_COLLISION_PROJECTOR_OPEN",
}
EXACT_KINETIC_RANK14_DELTA_1E2_FROM_28212124=True
G0_2622_PROVENANCE_AND_ROUTE_PASS=all([
 PARENT_2621_EXECUTED["source_exact"],PARENT_2621_EXECUTED["error_count"]==0,
 PARENT_2621_EXECUTED["interval_backend_pass"],
 PARENT_2621_EXECUTED["K14_Neumann_q"]>1,
 EXACT_KINETIC_RANK14_DELTA_1E2_FROM_28212124,
])
assert G0_2622_PROVENANCE_AND_ROUTE_PASS
print("G0_2622_PROVENANCE_AND_ROUTE_PASS =",G0_2622_PROVENANCE_AND_ROUTE_PASS)
print("KINETIC ROUTE = exact determinant-factor certificate inherited; Neumann route retired")

G0_2622_PROVENANCE_AND_ROUTE_PASS = True
KINETIC ROUTE = exact determinant-factor certificate inherited; Neumann route retired


## Niveau 1 — Représentants exacts des trois collisions

The simplex coordinates are converted directly into normalized directions:
\((\sqrt u,0,\sqrt{1-u})\) on the (xz) edge and
\((0,\sqrt v,\sqrt{1-v})\) on the (yz) edge.

In [37]:
u_cross_2622=sp.Rational(171256595517,866048000000)
u_c_v0_2622=(-sp.Rational(284832107,268295000)
 +sp.Rational(395941,36488120000)*sp.sqrt(sp.Integer(16205811493)))
v_c_u0_2622=(-sp.Rational(150509593748759,35641420508441)
 +sp.Rational(12815600,35641420508441)*sp.sqrt(sp.Integer(161292621824569)))

collision_directions={
 "q2_q3_xz":(sp.sqrt(u_cross_2622),sp.Integer(0),sp.sqrt(1-u_cross_2622)),
 "cubic_v0_xz":(sp.sqrt(u_c_v0_2622),sp.Integer(0),sp.sqrt(1-u_c_v0_2622)),
 "cubic_u0_yz":(sp.Integer(0),sp.sqrt(v_c_u0_2622),sp.sqrt(1-v_c_u0_2622)),
}
COLLISION_DIRECTION_DOMAIN_PASS=all([
 0<float(u_cross_2622)<1,0<float(u_c_v0_2622)<1,0<float(v_c_u0_2622)<1,
])
assert COLLISION_DIRECTION_DOMAIN_PASS
print({k:tuple(float(x) for x in v) for k,v in collision_directions.items()})

{'q2_q3_xz': (0.4446851959658293, 0.0, 0.8956869299531126), 'cubic_v0_xz': (0.5654609545157344, 0.0, 0.8247750656501168), 'cubic_u0_yz': (0.0, 0.5862567645642918, 0.8101253026554649)}


## Niveau 2 — Contours de Riesz centraux et budgets de transport

For each repeated spectral cluster, a circular contour is placed at one third
of the nearest distinct-cluster separation. The minimum singular value of
\(zI-A_{\rm phys}(c_0,\hat k)\) on 256 contour nodes is the numerical seed
for a validated resolvent lower bound. The trapezoidal Riesz integral is checked
for idempotence and integer trace. These are diagnostics until the entire
contour, rather than its nodes, is interval-enclosed.

In [38]:
def spectral_clusters(values,tol=2e-6):
    vals=sorted((complex(v) for v in values),key=lambda z:(z.real,z.imag))
    groups=[]
    for z in vals:
        if not groups or abs(z-np.mean(groups[-1]))>tol: groups.append([z])
        else: groups[-1].append(z)
    return groups

def riesz_seed(A,cluster_center,contour_radius,nodes=256):
    I=np.eye(A.shape[0]); P=np.zeros_like(A,dtype=complex); min_sv=np.inf
    for j in range(nodes):
        theta=2*np.pi*j/nodes; z=cluster_center+contour_radius*np.exp(1j*theta)
        R=np.linalg.inv(z*I-A)
        min_sv=min(min_sv,float(np.linalg.svd(z*I-A,compute_uv=False)[-1]))
        P += contour_radius*np.exp(1j*theta)*R/nodes
    return {"min_sampled_resolvent_sv":min_sv,
            "projector_idempotence_residual":float(np.linalg.norm(P@P-P,ord=2)),
            "projector_trace_real":float(np.trace(P).real),
            "projector_trace_imag":float(abs(np.trace(P).imag))}

riesz_seed_ledger=[]
center_coupling=(1.0,2.0,1.0)
for name,direction in collision_directions.items():
    obj=coupling_direction_bundle(center_coupling,tuple(float(x) for x in direction))
    A=obj["Aphys"]; groups=spectral_clusters(np.linalg.eigvals(A))
    centers=[complex(np.mean(g)) for g in groups]
    repeated=[(i,g) for i,g in enumerate(groups) if len(g)>1]
    for i,g in repeated:
        gap=min(abs(centers[i]-centers[j]) for j in range(len(groups)) if j!=i)
        radius=float(gap/3.0)
        seed=riesz_seed(A,centers[i],radius)
        row={"collision":name,"cluster_center_real":float(centers[i].real),
             "cluster_center_imag":float(centers[i].imag),"multiplicity":len(g),
             "nearest_distinct_gap":float(gap),"contour_radius":radius,**seed}
        row["numerical_seed_pass"]=all([
            seed["min_sampled_resolvent_sv"]>0,
            seed["projector_idempotence_residual"]<2e-6,
            abs(seed["projector_trace_real"]-len(g))<2e-5,
            seed["projector_trace_imag"]<2e-5,
        ])
        riesz_seed_ledger.append(row)

CENTRAL_RIESZ_CONTOUR_NUMERICAL_SEEDS_PASS=bool(riesz_seed_ledger) and all(r["numerical_seed_pass"] for r in riesz_seed_ledger)
print(pd.DataFrame(riesz_seed_ledger).to_string(index=False))
print("CENTRAL_RIESZ_CONTOUR_NUMERICAL_SEEDS_PASS =",CENTRAL_RIESZ_CONTOUR_NUMERICAL_SEEDS_PASS)

  collision  cluster_center_real  cluster_center_imag  multiplicity  nearest_distinct_gap  contour_radius  min_sampled_resolvent_sv  projector_idempotence_residual  projector_trace_real  projector_trace_imag  numerical_seed_pass
   q2_q3_xz            -1.402912         0.000000e+00             2              0.185652        0.061884                  0.002363                    1.065606e-13                   2.0          2.439082e-16                 True
   q2_q3_xz            -1.000000         0.000000e+00             5              0.090387        0.030129                  0.001770                    8.554424e-14                   5.0          3.595877e-16                 True
   q2_q3_xz             1.000000         0.000000e+00             5              0.090387        0.030129                  0.001770                    6.389134e-14                   5.0          2.175562e-16                 True
   q2_q3_xz             1.402912         0.000000e+00             2              0.1

## Niveau 3 — Inégalité validée requise

On a contour cell, let (m_Gamma) be an outward-certified lower bound for
\(\sigma_{\min}(zI-A_0)\) and (E) an outward-certified upper bound for
\(\|A-A_0\|_2\). If (E<m_Gamma), the contour remains in the resolvent set
throughout that cell and the Riesz rank is invariant. Both quantities must be
continuous interval bounds; sampled values cannot populate these flags.

In [39]:
RADIUS_CANDIDATE=1e-4
FULL_CONTOUR_OUTWARD_SIGMA_MIN_LOWER_BOUND_CERTIFIED=False
APHYS_COUPLING_DIRECTION_CELL_PERTURBATION_UPPER_BOUND_CERTIFIED=False
RIESZ_TRANSPORT_INEQUALITY_CERTIFIED=False
RIESZ_RANK_INVARIANT_ON_ALL_CELLS=False
TRANSPORTED_KREIN_FORM_INTERVAL_ENCLOSED=False
TRANSPORTED_KREIN_DEFINITE_TYPE_MARGIN_POSITIVE=False
UNIFORM_PROJECTOR_MARGIN_CERTIFIED=False

VALIDATED_RIESZ_LEDGER={
 "exact_kinetic_rank14_delta_1e2_inherited":EXACT_KINETIC_RANK14_DELTA_1E2_FROM_28212124,
 "central_Riesz_contour_numerical_seeds_pass":CENTRAL_RIESZ_CONTOUR_NUMERICAL_SEEDS_PASS,
 "full_contour_outward_sigma_min_lower_bound_certified":FULL_CONTOUR_OUTWARD_SIGMA_MIN_LOWER_BOUND_CERTIFIED,
 "Aphys_coupling_direction_cell_perturbation_upper_bound_certified":APHYS_COUPLING_DIRECTION_CELL_PERTURBATION_UPPER_BOUND_CERTIFIED,
 "Riesz_transport_inequality_certified":RIESZ_TRANSPORT_INEQUALITY_CERTIFIED,
 "Riesz_rank_invariant_on_all_cells":RIESZ_RANK_INVARIANT_ON_ALL_CELLS,
 "transported_Krein_form_interval_enclosed":TRANSPORTED_KREIN_FORM_INTERVAL_ENCLOSED,
 "transported_Krein_definite_type_margin_positive":TRANSPORTED_KREIN_DEFINITE_TYPE_MARGIN_POSITIVE,
 "uniform_projector_margin_certified":UNIFORM_PROJECTOR_MARGIN_CERTIFIED,
}
print(pd.DataFrame([{"gate":k,"pass":v} for k,v in VALIDATED_RIESZ_LEDGER.items()]).to_string(index=False))

                                                            gate  pass
                        exact_kinetic_rank14_delta_1e2_inherited  True
                      central_Riesz_contour_numerical_seeds_pass  True
            full_contour_outward_sigma_min_lower_bound_certified False
Aphys_coupling_direction_cell_perturbation_upper_bound_certified False
                            Riesz_transport_inequality_certified False
                               Riesz_rank_invariant_on_all_cells False
                        transported_Krein_form_interval_enclosed False
                 transported_Krein_definite_type_margin_positive False
                              uniform_projector_margin_certified False


## Niveau 4 — Verdict et prochain verrou

This step materially fixes the route and produces all central contour seeds,
but it does not yet promote nodewise resolvent margins to interval bounds.
The next step must enclose rational fixed-frame (A_{\rm phys}) over each
coupling-direction cell and certify the full contour arcs.

In [40]:
EXPLICIT_COUPLING_RADIUS_CERTIFIED=all([
 FULL_CONTOUR_OUTWARD_SIGMA_MIN_LOWER_BOUND_CERTIFIED,
 APHYS_COUPLING_DIRECTION_CELL_PERTURBATION_UPPER_BOUND_CERTIFIED,
 RIESZ_TRANSPORT_INEQUALITY_CERTIFIED,RIESZ_RANK_INVARIANT_ON_ALL_CELLS,
 TRANSPORTED_KREIN_FORM_INTERVAL_ENCLOSED,
 TRANSPORTED_KREIN_DEFINITE_TYPE_MARGIN_POSITIVE,
 UNIFORM_PROJECTOR_MARGIN_CERTIFIED,
])
CERTIFIED_COUPLING_RADIUS_LOWER_BOUND=(RADIUS_CANDIDATE if EXPLICIT_COUPLING_RADIUS_CERTIFIED else None)
FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN=False
FOUR_LEVEL_PROTOCOL_2622_PASS=all([
 G0_2622_PROVENANCE_AND_ROUTE_PASS,CENTRAL_RIESZ_CONTOUR_NUMERICAL_SEEDS_PASS,
 not EXPLICIT_COUPLING_RADIUS_CERTIFIED,
 not FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN,
])
assert FOUR_LEVEL_PROTOCOL_2622_PASS
STATUS_2622="PARTIAL_RIESZ_CONTOUR_SEEDS_PASS_INTERVAL_TRANSPORT_AND_KREIN_MARGIN_OPEN"
NEXT_AUTHORIZED=(
 ".28.21.2.1.2.6.2.2.1 — outward-rounded fixed-frame Aphys cell enclosure "
 "and full-contour resolvent lower bound"
)
verdict_2622={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.2_Validated_Riesz_Contour_Moving_Collision_Clusters_and_Transported_Krein_Margin_FAST",
 "parent":PARENT_2621_EXECUTED,"validated_route":VALIDATED_RIESZ_LEDGER,
 "central_Riesz_seed_ledger":riesz_seed_ledger,
 "explicit_coupling_radius_certified":False,"certified_coupling_radius_lower_bound":None,
 "full_delta_1e2_coupling_box_strong_hyperbolicity_proven":False,
 "status":STATUS_2622,"next_authorized":NEXT_AUTHORIZED,
}
export_dir=Path('/mnt/data/gvh_exports_2821212622'); export_dir.mkdir(parents=True,exist_ok=True)
verdict_path_2622=export_dir/'gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.2_riesz_transport_gate.json'
verdict_path_2622.write_text(json.dumps(verdict_2622,indent=2,ensure_ascii=False),encoding='utf-8')
print("STATUS =",STATUS_2622)
print("EXPLICIT_COUPLING_RADIUS_CERTIFIED =",EXPLICIT_COUPLING_RADIUS_CERTIFIED)
print("CERTIFIED_COUPLING_RADIUS_LOWER_BOUND =",CERTIFIED_COUPLING_RADIUS_LOWER_BOUND)
print("NEXT_AUTHORIZED =",NEXT_AUTHORIZED)
print("verdict JSON =",verdict_path_2622)

STATUS = PARTIAL_RIESZ_CONTOUR_SEEDS_PASS_INTERVAL_TRANSPORT_AND_KREIN_MARGIN_OPEN
EXPLICIT_COUPLING_RADIUS_CERTIFIED = False
CERTIFIED_COUPLING_RADIUS_LOWER_BOUND = None
NEXT_AUTHORIZED = .28.21.2.1.2.6.2.2.1 — outward-rounded fixed-frame Aphys cell enclosure and full-contour resolvent lower bound
verdict JSON = /mnt/data/gvh_exports_2821212622/gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.2_riesz_transport_gate.json


# GVH Diagonal Cubic `.28.21.2.1.2.6.2.2.1` — FAST

## Quantitative continuation of the exact `.2.3` projector and definite-type certificates

This notebook returns to the exact symmetric raw-pencil proof rather than
densifying the numerical (A_{\rm phys}) grid. It reconstructs the exact
two-dimensional physical collision quotient at each of the three canonical
collisions, extracts the exact Krein matrices and basis-normalized central
margins, and defines the strict variation budget needed for an explicit
coupling radius.

In [41]:
from sympy.polys.matrices import DomainMatrix

PARENT_2622_EXECUTED={
 "artifact":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.2_Validated_Riesz_Contour_Moving_Collision_Clusters_and_Transported_Krein_Margin_FAST(1).ipynb",
 "executed_size_bytes":141652,
 "executed_sha256":"971ed8b6aa0684a357f1c2661c49798c382e146556ec289b03fa0155e5f3470d",
 "source_exact":True,"code_cells":40,"error_count":0,
 "central_Riesz_contour_numerical_seeds_pass":True,
 "explicit_coupling_radius_certified":False,
}
PARENT_23_EXACT_CERTIFICATES={
 "light_projector_uniform_control_certified":True,
 "residual_collision_atlas_complete":True,
 "q2q3_definite_type_certified":True,
 "cubic_v0_definite_type_certified":True,
 "cubic_u0_definite_type_certified":True,
 "residual_projector_uniform_control_certified":True,
 "bounded_diagonalizer_certified":True,
 "strong_hyperbolicity_fixed_healthy_witness":True,
}
G0_26221_PROVENANCE_PASS=all([
 PARENT_2622_EXECUTED["source_exact"],PARENT_2622_EXECUTED["error_count"]==0,
 PARENT_2622_EXECUTED["central_Riesz_contour_numerical_seeds_pass"],
 not PARENT_2622_EXECUTED["explicit_coupling_radius_certified"],
 all(PARENT_23_EXACT_CERTIFICATES.values()),
])
assert G0_26221_PROVENANCE_PASS
print("G0_26221_PROVENANCE_PASS =",G0_26221_PROVENANCE_PASS)

G0_26221_PROVENANCE_PASS = True


## Niveau 1 — Reconstruction exacte des formes de collision

The routine follows the exact `.2.3` quotient construction. Gauge directions
are proved to lie in the radical of (W=\partial_\omega P); two exact kernel
directions complete the physical collision quotient. Both the Krein matrix and
its Euclidean coordinate Gram matrix are retained.

In [42]:
def _dm(M,K): return DomainMatrix.from_Matrix(M).convert_to(K)

def exact_collision_quantitative_data(u0,v0,y0,edge):
    if edge=="xz": kvec=(sp.sqrt(sp.simplify(u0/(1-u0))),sp.Integer(0),sp.Integer(1))
    elif edge=="yz": kvec=(sp.Integer(0),sp.sqrt(sp.simplify(v0/(1-v0))),sp.Integer(1))
    else: raise ValueError("edge")
    r2=sp.simplify(sum(x*x for x in kvec)); omega=sp.sqrt(sp.simplify(y0*r2))
    qphys=kvec[0] if edge=="xz" else kvec[1]
    Kfield=sp.QQ.algebraic_field(qphys,omega)
    Kdm=_dm(K_h,Kfield); Mdm={i:_dm(M_h[i],Kfield) for i in (1,2,3)}
    Gdm={key:_dm(G_h[key],Kfield) for key in G_h}
    xyz=[Kfield.from_sympy(x) for x in kvec]; om=Kfield.from_sympy(omega); two=Kfield.convert(2)
    B=Mdm[1]*xyz[0]+Mdm[2]*xyz[1]+Mdm[3]*xyz[2]
    C=(Gdm[(1,1)]*(xyz[0]*xyz[0])+Gdm[(2,2)]*(xyz[1]*xyz[1])+Gdm[(3,3)]*(xyz[2]*xyz[2])
       +Gdm[(1,2)]*(two*xyz[0]*xyz[1])+Gdm[(1,3)]*(two*xyz[0]*xyz[2])+Gdm[(2,3)]*(two*xyz[1]*xyz[2]))
    P=Kdm*(om*om)+B*om+C; W=Kdm*(two*om)+B; N=P.nullspace()
    Udiff=sp.Matrix.hstack(*original_gauge_vectors((omega,*kvec))[:4]).subs(healthy_subs)
    U6=_dm(sp.Matrix.hstack(Udiff,N_trace,N_radial),Kfield)
    assert (P*U6).is_zero_matrix and (U6.transpose()*W*N.transpose()).is_zero_matrix
    current=U6.transpose(); rank=current.rank(); selected=[]
    for i in range(N.shape[0]):
        row=N.extract([i],list(range(20))); trial=DomainMatrix.vstack(current,row)
        if trial.rank()>rank: selected.append(i); current=trial; rank=trial.rank()
        if rank==8: break
    E=N.extract(selected,list(range(20))).transpose()
    GW=(E.transpose()*W*E).to_Matrix(); GE=(E.transpose()*E).to_Matrix()
    assert len(selected)==2 and GW.is_symmetric() and GE.is_symmetric()
    assert GW.is_positive_definite and GE.is_positive_definite
    return {"kvec":kvec,"omega":omega,"selected":selected,"GW":GW,"GE":GE,
            "gauge_radical":True,"physical_dim":2}

y_cross_26221=sp.Rational(26767,13600)
y_c_v0_26221=(sp.Rational(1035103,1395134)+sp.Rational(133,23717278)*sp.sqrt(sp.Integer(16205811493)))
y_c_u0_26221=(sp.Rational(4021300195103,42121678782703)+sp.Rational(5625200,42121678782703)*sp.sqrt(sp.Integer(161292621824569)))
collision_exact_inputs={
 "q2_q3":(u_cross_2622,sp.Integer(0),y_cross_26221,"xz"),
 "cubic_v0":(u_c_v0_2622,sp.Integer(0),y_c_v0_26221,"xz"),
 "cubic_u0":(sp.Integer(0),v_c_u0_2622,y_c_u0_26221,"yz"),
}
exact_collision_data={k:exact_collision_quantitative_data(*v) for k,v in collision_exact_inputs.items()}
EXACT_CENTRAL_COLLISION_QUOTIENTS_RECONSTRUCTED=True
print("EXACT_CENTRAL_COLLISION_QUOTIENTS_RECONSTRUCTED =",EXACT_CENTRAL_COLLISION_QUOTIENTS_RECONSTRUCTED)

TypeError: 'Fraction' object is not subscriptable

## Niveau 2 — Marges centrales normalisées

The coordinate-invariant central margin is the smallest generalized eigenvalue
of (G_Wx=\mu G_Ex). Exact positivity follows from the exact matrices; the
displayed decimal value is used only to size the continuation budget.

In [ ]:
central_type_margin_ledger=[]
for name,d in exact_collision_data.items():
    GW=np.asarray(d["GW"].evalf(40),dtype=float); GE=np.asarray(d["GE"].evalf(40),dtype=float)
    H=np.linalg.solve(GE,GW); mu=np.linalg.eigvals(H).real
    row={"collision":name,"physical_dim":d["physical_dim"],"selected_rows":d["selected"],
         "GW_offdiag_exact_zero":bool(sp.simplify(d["GW"][0,1])==0),
         "GW_positive_definite_exact":bool(d["GW"].is_positive_definite),
         "GE_positive_definite_exact":bool(d["GE"].is_positive_definite),
         "normalized_type_margin_numeric":float(np.min(mu)),
         "normalized_type_max_numeric":float(np.max(mu))}
    row["pass"]=all([row["GW_positive_definite_exact"],row["GE_positive_definite_exact"],
                      row["normalized_type_margin_numeric"]>0])
    central_type_margin_ledger.append(row)
CENTRAL_EXACT_DEFINITE_TYPE_AND_NUMERIC_MARGIN_PASS=all(r["pass"] for r in central_type_margin_ledger)
assert CENTRAL_EXACT_DEFINITE_TYPE_AND_NUMERIC_MARGIN_PASS
CENTRAL_MIN_NORMALIZED_TYPE_MARGIN=min(r["normalized_type_margin_numeric"] for r in central_type_margin_ledger)
print(pd.DataFrame(central_type_margin_ledger).to_string(index=False))
print("CENTRAL_MIN_NORMALIZED_TYPE_MARGIN =",CENTRAL_MIN_NORMALIZED_TYPE_MARGIN)
print("CENTRAL_EXACT_DEFINITE_TYPE_AND_NUMERIC_MARGIN_PASS =",CENTRAL_EXACT_DEFINITE_TYPE_AND_NUMERIC_MARGIN_PASS)

## Niveau 3 — Budget de continuation quantitative

To preserve definite type it suffices to certify, in transported orthonormal
coordinates, a variation strictly smaller than the central normalized margin.
The current notebook extracts that margin exactly/numerically but does not yet
possess an interval enclosure of the transported eigenspaces. Hence the
Lipschitz constant and released radius remain unset rather than estimated from
finite differences.

In [ ]:
TYPE_MARGIN_SAFETY_FRACTION=0.5
ALLOWED_TRANSPORTED_FORM_VARIATION=(TYPE_MARGIN_SAFETY_FRACTION*CENTRAL_MIN_NORMALIZED_TYPE_MARGIN)
TRANSPORTED_COLLISION_BUNDLE_INTERVAL_FRAME_CERTIFIED=False
TRANSPORTED_KREIN_FORM_VARIATION_BOUND_CERTIFIED=False
TYPE_MARGIN_LIPSCHITZ_CONSTANT_CERTIFIED=None
TYPE_RADIUS_LOWER_BOUND=None
REGULAR_REGION_COUPLING_GAP_VARIATION_CERTIFIED=False
LIGHT_RESIDUAL_COUPLING_SEPARATION_CERTIFIED=False
UNIFORM_PROJECTOR_CONTINUATION_BOUND_CERTIFIED=False

CONTINUATION_BUDGET_LEDGER={
 "central_exact_definite_type_and_numeric_margin_pass":CENTRAL_EXACT_DEFINITE_TYPE_AND_NUMERIC_MARGIN_PASS,
 "central_min_normalized_type_margin":CENTRAL_MIN_NORMALIZED_TYPE_MARGIN,
 "allowed_transported_form_variation":ALLOWED_TRANSPORTED_FORM_VARIATION,
 "transported_collision_bundle_interval_frame_certified":TRANSPORTED_COLLISION_BUNDLE_INTERVAL_FRAME_CERTIFIED,
 "transported_Krein_form_variation_bound_certified":TRANSPORTED_KREIN_FORM_VARIATION_BOUND_CERTIFIED,
 "type_margin_Lipschitz_constant_certified":TYPE_MARGIN_LIPSCHITZ_CONSTANT_CERTIFIED,
 "type_radius_lower_bound":TYPE_RADIUS_LOWER_BOUND,
 "regular_region_coupling_gap_variation_certified":REGULAR_REGION_COUPLING_GAP_VARIATION_CERTIFIED,
 "light_residual_coupling_separation_certified":LIGHT_RESIDUAL_COUPLING_SEPARATION_CERTIFIED,
 "uniform_projector_continuation_bound_certified":UNIFORM_PROJECTOR_CONTINUATION_BOUND_CERTIFIED,
}
print(json.dumps(CONTINUATION_BUDGET_LEDGER,indent=2))

## Niveau 4 — Verdict borné et prochaine cible

This step recovers quantitative central margins directly from the exact `.2.3`
collision proof. It is a stronger starting point than a sampled contour alone,
but it is not yet an open-coupling theorem. The remaining task is now sharply
defined: validated transport of the two-dimensional quotient frames and an
outward-rounded bound on their Krein-form variation.

In [ ]:
COUPLING_DEPENDENT_DEFINITE_TYPE_CERTIFIED=all([
 TRANSPORTED_COLLISION_BUNDLE_INTERVAL_FRAME_CERTIFIED,
 TRANSPORTED_KREIN_FORM_VARIATION_BOUND_CERTIFIED,
 TYPE_MARGIN_LIPSCHITZ_CONSTANT_CERTIFIED is not None,
 TYPE_RADIUS_LOWER_BOUND is not None,
])
EXPLICIT_COUPLING_RADIUS_CERTIFIED=all([
 COUPLING_DEPENDENT_DEFINITE_TYPE_CERTIFIED,
 REGULAR_REGION_COUPLING_GAP_VARIATION_CERTIFIED,
 LIGHT_RESIDUAL_COUPLING_SEPARATION_CERTIFIED,
 UNIFORM_PROJECTOR_CONTINUATION_BOUND_CERTIFIED,
])
CERTIFIED_COUPLING_RADIUS_LOWER_BOUND=(TYPE_RADIUS_LOWER_BOUND if EXPLICIT_COUPLING_RADIUS_CERTIFIED else None)
FULL_DELTA_1E2_COUPLING_BOX_STRONG_HYPERBOLICITY_PROVEN=False
FOUR_LEVEL_PROTOCOL_26221_PASS=all([
 G0_26221_PROVENANCE_PASS,EXACT_CENTRAL_COLLISION_QUOTIENTS_RECONSTRUCTED,
 CENTRAL_EXACT_DEFINITE_TYPE_AND_NUMERIC_MARGIN_PASS,
 not EXPLICIT_COUPLING_RADIUS_CERTIFIED,
])
assert FOUR_LEVEL_PROTOCOL_26221_PASS
STATUS_26221="PASS_EXACT_CENTRAL_TYPE_MARGINS_EXTRACTED_TRANSPORT_VARIATION_BOUND_OPEN"
NEXT_AUTHORIZED=(
 ".28.21.2.1.2.6.2.2.2 — validated transport of exact collision quotient frames "
 "and outward-rounded Krein-form Lipschitz bound"
)
verdict_26221={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.2.1_Quantitative_Continuation_of_Exact_2.3_Projector_and_Definite_Type_Certificates_FAST",
 "parents":{"riesz_seed":PARENT_2622_EXECUTED,"exact_2.3":PARENT_23_EXACT_CERTIFICATES},
 "central_type_margin_ledger":central_type_margin_ledger,
 "continuation_budget":CONTINUATION_BUDGET_LEDGER,
 "coupling_dependent_definite_type_certified":False,
 "explicit_coupling_radius_certified":False,"certified_coupling_radius_lower_bound":None,
 "full_delta_1e2_coupling_box_strong_hyperbolicity_proven":False,
 "status":STATUS_26221,"next_authorized":NEXT_AUTHORIZED,
}
export_dir=Path('/mnt/data/gvh_exports_28212126221'); export_dir.mkdir(parents=True,exist_ok=True)
verdict_path_26221=export_dir/'gvh_0.3.2.7.3.7.3.3.28.21.2.1.2.6.2.2.1_exact_type_margin_continuation.json'
verdict_path_26221.write_text(json.dumps(verdict_26221,indent=2,ensure_ascii=False),encoding='utf-8')
print("STATUS =",STATUS_26221)
print("COUPLING_DEPENDENT_DEFINITE_TYPE_CERTIFIED =",COUPLING_DEPENDENT_DEFINITE_TYPE_CERTIFIED)
print("EXPLICIT_COUPLING_RADIUS_CERTIFIED =",EXPLICIT_COUPLING_RADIUS_CERTIFIED)
print("NEXT_AUTHORIZED =",NEXT_AUTHORIZED)
print("verdict JSON =",verdict_path_26221)